In [1]:
# ============================================================
# AutoML Pipeline Components and SHAP Variation
# Revised Experimental Framework
#
# Phase 1–3: Workshop-paper scope
# Items 1–9 + 11
# ============================================================

import os
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Reproducibility
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# ------------------------------------------------------------
# Experiment scope
# ------------------------------------------------------------

DATASET_IDS = [37, 31, 44, 4534, 1461]

# Controlled factorial factors
N_CONFIGS_PER_CELL = 5       # 5–10 according to the methodology
N_CV_FOLDS = 5

# Replicate structure for noise floor
N_TRAINING_SEEDS = 3
N_EXPLAINER_REPLICATES = 5

# SHAP configuration
BACKGROUND_SIZE = 100
EXPLANATION_SIZE = 200
SHAP_MAX_EVALS = 1000

# Rashomon sensitivity
RASHOMON_SE_MULTIPLIERS = [1.0, 2.0]

# PERMANOVA
N_PERMUTATIONS = 9999

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results_new_experiment"
CONFIG_DIR = RESULTS_DIR / "configurations"
MODEL_DIR = RESULTS_DIR / "models"
SHAP_DIR = RESULTS_DIR / "shap"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
FIGURE_DIR = RESULTS_DIR / "figures"

for directory in [
    RESULTS_DIR,
    CONFIG_DIR,
    MODEL_DIR,
    SHAP_DIR,
    ANALYSIS_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Datasets:", DATASET_IDS)
print("Configurations per cell:", N_CONFIGS_PER_CELL)
print("CV folds:", N_CV_FOLDS)
print("Training seeds:", N_TRAINING_SEEDS)
print("Explainer replicates:", N_EXPLAINER_REPLICATES)

Project root: /home/dbanerjee/example_for_debjeet
Datasets: [37, 31, 44, 4534, 1461]
Configurations per cell: 5
CV folds: 5
Training seeds: 3
Explainer replicates: 5


In [2]:
# ============================================================
# CELL 2 — Auto-sklearn configuration space
# ============================================================

import autosklearn
from autosklearn.pipeline.components.classification import ClassifierChoice

print("Auto-sklearn version:", autosklearn.__version__)

# Create the classifier-choice component
classifier_choice = ClassifierChoice(
    dataset_properties={},
    feat_type=None
)

# Obtain Auto-sklearn's classification configuration space
cs = classifier_choice.get_hyperparameter_search_space(
    feat_type=None,
    dataset_properties={}
)

print("\nConfiguration space:")
print(cs)

print("\nNumber of hyperparameters:", len(cs.get_hyperparameters()))

Auto-sklearn version: 0.15.0

Configuration space:
Configuration space object:
  Hyperparameters:
    __choice__, Type: Categorical, Choices: {adaboost, bernoulli_nb, decision_tree, extra_trees, gaussian_nb, gradient_boosting, k_nearest_neighbors, lda, liblinear_svc, libsvm_svc, mlp, multinomial_nb, passive_aggressive, qda, random_forest, sgd}, Default: random_forest
    adaboost:algorithm, Type: Categorical, Choices: {SAMME.R, SAMME}, Default: SAMME.R
    adaboost:learning_rate, Type: UniformFloat, Range: [0.01, 2.0], Default: 0.1, on log-scale
    adaboost:max_depth, Type: UniformInteger, Range: [1, 10], Default: 1
    adaboost:n_estimators, Type: UniformInteger, Range: [50, 500], Default: 50
    bernoulli_nb:alpha, Type: UniformFloat, Range: [0.01, 100.0], Default: 1.0, on log-scale
    bernoulli_nb:fit_prior, Type: Categorical, Choices: {True, False}, Default: True
    decision_tree:criterion, Type: Categorical, Choices: {gini, entropy}, Default: gini
    decision_tree:max_depth_fa

In [3]:
# ============================================================
# CELL 3 — Inspect all Auto-sklearn pipeline component spaces
# ============================================================

from autosklearn.pipeline.components.classification import (
    ClassifierChoice
)

from autosklearn.pipeline.components.feature_preprocessing import (
    FeaturePreprocessorChoice
)

from autosklearn.pipeline.components.data_preprocessing import (
    DataPreprocessorChoice
)

print("=" * 70)
print("CLASSIFIERS")
print("=" * 70)

classifier_components = ClassifierChoice.get_components()

print("Number of classifiers:", len(classifier_components))
print("Classifiers:")
for name in sorted(classifier_components):
    print("  ", name)


print("\n" + "=" * 70)
print("FEATURE PREPROCESSORS")
print("=" * 70)

feature_preprocessor_components = FeaturePreprocessorChoice.get_components()

print("Number of feature preprocessors:", len(feature_preprocessor_components))
print("Feature preprocessors:")
for name in sorted(feature_preprocessor_components):
    print("  ", name)


print("\n" + "=" * 70)
print("DATA PREPROCESSORS")
print("=" * 70)

data_preprocessor_components = DataPreprocessorChoice.get_components()

print("Number of data preprocessors:", len(data_preprocessor_components))
print("Data preprocessors:")
for name in sorted(data_preprocessor_components):
    print("  ", name)

CLASSIFIERS
Number of classifiers: 16
Classifiers:
   adaboost
   bernoulli_nb
   decision_tree
   extra_trees
   gaussian_nb
   gradient_boosting
   k_nearest_neighbors
   lda
   liblinear_svc
   libsvm_svc
   mlp
   multinomial_nb
   passive_aggressive
   qda
   random_forest
   sgd

FEATURE PREPROCESSORS
Number of feature preprocessors: 18
Feature preprocessors:
   densifier
   extra_trees_preproc_for_classification
   extra_trees_preproc_for_regression
   fast_ica
   feature_agglomeration
   kernel_pca
   kitchen_sinks
   liblinear_svc_preprocessor
   no_preprocessing
   nystroem_sampler
   pca
   polynomial
   random_trees_embedding
   select_percentile_classification
   select_percentile_regression
   select_rates_classification
   select_rates_regression
   truncatedSVD

DATA PREPROCESSORS
Number of data preprocessors: 1
Data preprocessors:
   feature_type


In [4]:
# ============================================================
# CELL 4 — Auto-sklearn Classification Configuration Space
# ============================================================

from autosklearn.pipeline.components.classification import ClassifierChoice

# Create ClassifierChoice instance
classifier_choice = ClassifierChoice(
    dataset_properties={},
    feat_type=None
)

# Get configuration space
pipeline_cs = classifier_choice.get_hyperparameter_search_space(
    feat_type=None,
    dataset_properties={}
)

print("=" * 80)
print("AUTO-SKLEARN CLASSIFICATION CONFIGURATION SPACE")
print("=" * 80)

print(pipeline_cs)

print("\nNumber of hyperparameters:",
      len(pipeline_cs.get_hyperparameters()))

print("\n" + "=" * 80)
print("FORBIDDEN CLAUSES")
print("=" * 80)

for forbidden in pipeline_cs.get_forbiddens():
    print(forbidden)

print("\n" + "=" * 80)
print("CONDITIONS")
print("=" * 80)

for condition in pipeline_cs.get_conditions():
    print(condition)

AUTO-SKLEARN CLASSIFICATION CONFIGURATION SPACE
Configuration space object:
  Hyperparameters:
    __choice__, Type: Categorical, Choices: {adaboost, bernoulli_nb, decision_tree, extra_trees, gaussian_nb, gradient_boosting, k_nearest_neighbors, lda, liblinear_svc, libsvm_svc, mlp, multinomial_nb, passive_aggressive, qda, random_forest, sgd}, Default: random_forest
    adaboost:algorithm, Type: Categorical, Choices: {SAMME.R, SAMME}, Default: SAMME.R
    adaboost:learning_rate, Type: UniformFloat, Range: [0.01, 2.0], Default: 0.1, on log-scale
    adaboost:max_depth, Type: UniformInteger, Range: [1, 10], Default: 1
    adaboost:n_estimators, Type: UniformInteger, Range: [50, 500], Default: 50
    bernoulli_nb:alpha, Type: UniformFloat, Range: [0.01, 100.0], Default: 1.0, on log-scale
    bernoulli_nb:fit_prior, Type: Categorical, Choices: {True, False}, Default: True
    decision_tree:criterion, Type: Categorical, Choices: {gini, entropy}, Default: gini
    decision_tree:max_depth_facto

In [5]:
# ============================================================
# CELL 5 — REAL SimpleClassificationPipeline CONFIG SPACE
# ============================================================

from autosklearn.pipeline.classification import SimpleClassificationPipeline

pipeline = SimpleClassificationPipeline()

real_pipeline_cs = pipeline.get_hyperparameter_search_space()

print("=" * 80)
print("SIMPLE CLASSIFICATION PIPELINE CONFIGURATION SPACE")
print("=" * 80)

print(real_pipeline_cs)

print("\nNumber of hyperparameters:",
      len(real_pipeline_cs.get_hyperparameters()))

print("\n" + "=" * 80)
print("HYPERPARAMETER NAMES")
print("=" * 80)

for hp in real_pipeline_cs.get_hyperparameters():
    print(hp.name)

SIMPLE CLASSIFICATION PIPELINE CONFIGURATION SPACE
Configuration space object:
  Hyperparameters:
    balancing:strategy, Type: Categorical, Choices: {none, weighting}, Default: none
    classifier:__choice__, Type: Categorical, Choices: {adaboost, bernoulli_nb, decision_tree, extra_trees, gaussian_nb, gradient_boosting, k_nearest_neighbors, lda, liblinear_svc, libsvm_svc, mlp, multinomial_nb, passive_aggressive, qda, random_forest, sgd}, Default: random_forest
    classifier:adaboost:algorithm, Type: Categorical, Choices: {SAMME.R, SAMME}, Default: SAMME.R
    classifier:adaboost:learning_rate, Type: UniformFloat, Range: [0.01, 2.0], Default: 0.1, on log-scale
    classifier:adaboost:max_depth, Type: UniformInteger, Range: [1, 10], Default: 1
    classifier:adaboost:n_estimators, Type: UniformInteger, Range: [50, 500], Default: 50
    classifier:bernoulli_nb:alpha, Type: UniformFloat, Range: [0.01, 100.0], Default: 1.0, on log-scale
    classifier:bernoulli_nb:fit_prior, Type: Categor

In [6]:
# ============================================================
# CELL 6 — CONTROLLED FACTORIAL DESIGN
# ============================================================

# Use the COMPLETE SimpleClassificationPipeline space
# created in Cell 5.
full_cs = real_pipeline_cs

# ------------------------------------------------------------
# Classifier factor
# ------------------------------------------------------------

classifier_hp = full_cs.get_hyperparameter(
    "classifier:__choice__"
)

CLASSIFIERS = sorted(classifier_hp.choices)

# ------------------------------------------------------------
# Feature-preprocessor factor
# ------------------------------------------------------------

feature_preprocessor_hp = full_cs.get_hyperparameter(
    "feature_preprocessor:__choice__"
)

FEATURE_PREPROCESSORS = sorted(
    feature_preprocessor_hp.choices
)

# ------------------------------------------------------------
# Balancing factor
# ------------------------------------------------------------

balancing_hp = full_cs.get_hyperparameter(
    "balancing:strategy"
)

BALANCING_STRATEGIES = sorted(
    balancing_hp.choices
)

# ------------------------------------------------------------
# Print design
# ------------------------------------------------------------

print("=" * 70)
print("CONTROLLED FACTORIAL")
print("=" * 70)

print("\nClassifiers:", len(CLASSIFIERS))
for x in CLASSIFIERS:
    print("  ", x)

print("\nFeature preprocessors:", len(FEATURE_PREPROCESSORS))
for x in FEATURE_PREPROCESSORS:
    print("  ", x)

print("\nBalancing strategies:", len(BALANCING_STRATEGIES))
for x in BALANCING_STRATEGIES:
    print("  ", x)

raw_factorial_size = (
    len(CLASSIFIERS)
    * len(FEATURE_PREPROCESSORS)
    * len(BALANCING_STRATEGIES)
)

print("\nRaw factorial size:", raw_factorial_size)

CONTROLLED FACTORIAL

Classifiers: 16
   adaboost
   bernoulli_nb
   decision_tree
   extra_trees
   gaussian_nb
   gradient_boosting
   k_nearest_neighbors
   lda
   liblinear_svc
   libsvm_svc
   mlp
   multinomial_nb
   passive_aggressive
   qda
   random_forest
   sgd

Feature preprocessors: 13
   extra_trees_preproc_for_classification
   fast_ica
   feature_agglomeration
   kernel_pca
   kitchen_sinks
   liblinear_svc_preprocessor
   no_preprocessing
   nystroem_sampler
   pca
   polynomial
   random_trees_embedding
   select_percentile_classification
   select_rates_classification

Balancing strategies: 2
   none
   weighting

Raw factorial size: 416


In [7]:
# ============================================================
# CELL 7 — STRUCTURAL FACTORIAL CELLS + FORBIDDEN HOLES
# ============================================================

FACTOR_NAMES = {
    "classifier:__choice__",
    "feature_preprocessor:__choice__",
    "balancing:strategy",
}


def get_literal_clauses(forbidden):
    """
    Extract the individual literal clauses from a ConfigSpace
    ForbiddenAndConjunction.

    ConfigSpace 0.15.x exposes these through:
        get_descendant_literal_clauses()
    """

    if hasattr(forbidden, "get_descendant_literal_clauses"):
        return list(
            forbidden.get_descendant_literal_clauses()
        )

    # Fallback for other ConfigSpace representations
    if hasattr(forbidden, "components"):
        return list(forbidden.components)

    return [forbidden]


def get_clause_info(clause):
    """
    Safely extract:
        hyperparameter name
        forbidden value
    """

    hp = getattr(clause, "hyperparameter", None)

    if hp is None:
        return None, None

    hp_name = getattr(hp, "name", None)
    hp_value = getattr(clause, "value", None)

    return hp_name, hp_value


# ============================================================
# EXTRACT STRUCTURAL CLASSIFIER × PREPROCESSOR PAIRS
# ============================================================

STRUCTURAL_FORBIDDEN_PAIRS = set()

for forbidden in full_cs.get_forbiddens():

    clauses = get_literal_clauses(forbidden)

    clause_info = []

    for clause in clauses:

        hp_name, hp_value = get_clause_info(clause)

        if hp_name is not None:
            clause_info.append(
                (hp_name, hp_value)
            )

    classifier_value = None
    preprocessor_value = None

    for hp_name, hp_value in clause_info:

        if hp_name == "classifier:__choice__":
            classifier_value = hp_value

        elif hp_name == "feature_preprocessor:__choice__":
            preprocessor_value = hp_value

    # A structural classifier × preprocessor hole
    # is completely determined by these two factors.
    if (
        classifier_value is not None
        and preprocessor_value is not None
    ):
        STRUCTURAL_FORBIDDEN_PAIRS.add(
            (
                classifier_value,
                preprocessor_value
            )
        )


# ============================================================
# DISPLAY STRUCTURAL HOLES
# ============================================================

print("=" * 70)
print("STRUCTURAL CLASSIFIER × PREPROCESSOR CONSTRAINTS")
print("=" * 70)

print(
    f"Detected forbidden classifier-preprocessor pairs: "
    f"{len(STRUCTURAL_FORBIDDEN_PAIRS)}"
)

for classifier, preprocessor in sorted(
    STRUCTURAL_FORBIDDEN_PAIRS
):
    print(
        f"  {classifier:30s} × {preprocessor}"
    )


# ============================================================
# ENUMERATE FULL FACTORIAL DESIGN
# ============================================================

valid_cells = []
forbidden_cells = []

for classifier in CLASSIFIERS:

    for preprocessor in FEATURE_PREPROCESSORS:

        for balancing in BALANCING_STRATEGIES:

            is_forbidden = (
                classifier,
                preprocessor
            ) in STRUCTURAL_FORBIDDEN_PAIRS

            cell = {
                "classifier": classifier,
                "feature_preprocessor": preprocessor,
                "balancing": balancing,
            }

            if is_forbidden:

                cell["reason"] = (
                    "classifier × feature_preprocessor "
                    "structural incompatibility"
                )

                forbidden_cells.append(cell)

            else:

                valid_cells.append(cell)


# ============================================================
# DATAFRAMES
# ============================================================

valid_cells_df = pd.DataFrame(
    valid_cells,
    columns=[
        "classifier",
        "feature_preprocessor",
        "balancing"
    ]
)

forbidden_cells_df = pd.DataFrame(
    forbidden_cells,
    columns=[
        "classifier",
        "feature_preprocessor",
        "balancing",
        "reason"
    ]
)


# ============================================================
# SUMMARY
# ============================================================

theoretical_cells = (
    len(CLASSIFIERS)
    * len(FEATURE_PREPROCESSORS)
    * len(BALANCING_STRATEGIES)
)

print("\n" + "=" * 70)
print("STRUCTURAL FACTORIAL DESIGN")
print("=" * 70)

print(f"Theoretical cells : {theoretical_cells}")
print(f"Valid cells       : {len(valid_cells_df)}")
print(f"Forbidden cells   : {len(forbidden_cells_df)}")

print("\nValid cells by classifier:")

display(
    valid_cells_df
    .groupby("classifier")
    .size()
    .sort_values()
    .rename("valid_cells")
    .to_frame()
)

print("\nForbidden structural cells:")

display(forbidden_cells_df)

STRUCTURAL CLASSIFIER × PREPROCESSOR CONSTRAINTS
Detected forbidden classifier-preprocessor pairs: 43
  adaboost                       × kernel_pca
  adaboost                       × kitchen_sinks
  adaboost                       × nystroem_sampler
  decision_tree                  × kernel_pca
  decision_tree                  × kitchen_sinks
  decision_tree                  × nystroem_sampler
  extra_trees                    × kernel_pca
  extra_trees                    × kitchen_sinks
  extra_trees                    × nystroem_sampler
  gaussian_nb                    × kernel_pca
  gaussian_nb                    × kitchen_sinks
  gaussian_nb                    × nystroem_sampler
  gaussian_nb                    × random_trees_embedding
  gradient_boosting              × kernel_pca
  gradient_boosting              × kitchen_sinks
  gradient_boosting              × nystroem_sampler
  gradient_boosting              × random_trees_embedding
  k_nearest_neighbors            × kernel_pca
 

,valid_cells
classifier,
multinomial_nb,2
gaussian_nb,18
gradient_boosting,18
adaboost,20
decision_tree,20
extra_trees,20
k_nearest_neighbors,20
libsvm_svc,20
mlp,20



Forbidden structural cells:


,classifier,feature_preprocessor,balancing,reason
0,adaboost,kernel_pca,none,classifier × feature_preprocessor structural i...
1,adaboost,kernel_pca,weighting,classifier × feature_preprocessor structural i...
2,adaboost,kitchen_sinks,none,classifier × feature_preprocessor structural i...
3,adaboost,kitchen_sinks,weighting,classifier × feature_preprocessor structural i...
4,adaboost,nystroem_sampler,none,classifier × feature_preprocessor structural i...
...,...,...,...,...
81,random_forest,kernel_pca,weighting,classifier × feature_preprocessor structural i...
82,random_forest,kitchen_sinks,none,classifier × feature_preprocessor structural i...
83,random_forest,kitchen_sinks,weighting,classifier × feature_preprocessor structural i...
84,random_forest,nystroem_sampler,none,classifier × feature_preprocessor structural i...


In [8]:
# ============================================================
# CELL 8 — REGENERATE CONFIGURATIONS WITH PINNED PREPROCESSING
# ============================================================

from collections import defaultdict


# ============================================================
# PINNED DATA-PREPROCESSOR SETTINGS
# ============================================================

PINNED_DATA_PREPROCESSOR = {
    "data_preprocessor:__choice__":
        "feature_type",

    "data_preprocessor:feature_type:"
    "categorical_transformer:categorical_encoding:__choice__":
        "one_hot_encoding",

    "data_preprocessor:feature_type:"
    "categorical_transformer:category_coalescence:__choice__":
        "minority_coalescer",

    "data_preprocessor:feature_type:"
    "numerical_transformer:imputation:strategy":
        "mean",

    "data_preprocessor:feature_type:"
    "numerical_transformer:rescaling:__choice__":
        "standardize",
}


# ============================================================
# VALID FACTORIAL CELLS
# ============================================================

VALID_CELL_SET = {
    (
        row["classifier"],
        row["feature_preprocessor"],
        row["balancing"]
    )
    for _, row in valid_cells_df.iterrows()
}


# ============================================================
# TARGET
# ============================================================

TARGET_PER_CELL = N_CONFIGS_PER_CELL

TOTAL_TARGET = (
    len(VALID_CELL_SET)
    * TARGET_PER_CELL
)


# ============================================================
# STORAGE
# ============================================================

cell_configurations = defaultdict(list)

# Prevent accidental duplicate configurations
cell_configuration_signatures = defaultdict(set)


# ============================================================
# CHECK WHETHER CONFIGURATION MATCHES PINNED PREPROCESSING
# ============================================================

def matches_pinned_preprocessing(config):

    values = config.get_dictionary()

    for hp_name, expected_value in (
        PINNED_DATA_PREPROCESSOR.items()
    ):

        actual_value = values.get(hp_name)

        if actual_value != expected_value:
            return False

    return True


# ============================================================
# CHECK WHETHER CONFIGURATION BELONGS TO VALID FACTORIAL CELL
# ============================================================

def get_factorial_cell(config):

    values = config.get_dictionary()

    classifier = values.get(
        "classifier:__choice__"
    )

    preprocessor = values.get(
        "feature_preprocessor:__choice__"
    )

    balancing = values.get(
        "balancing:strategy"
    )

    cell_key = (
        classifier,
        preprocessor,
        balancing
    )

    if cell_key not in VALID_CELL_SET:
        return None

    return cell_key


# ============================================================
# SAMPLE NATIVE CONFIGURATIONS
# ============================================================

print("=" * 70)
print("REGENERATING CONTROLLED CONFIGURATION POPULATION")
print("=" * 70)

print(
    f"Valid factorial cells : "
    f"{len(VALID_CELL_SET)}"
)

print(
    f"Target per cell       : "
    f"{TARGET_PER_CELL}"
)

print(
    f"Total target          : "
    f"{TOTAL_TARGET}"
)

print("\nPinned data preprocessing:")

for key, value in PINNED_DATA_PREPROCESSOR.items():
    print(f"  {key} = {value}")


# ============================================================
# SAMPLING LOOP
# ============================================================

sampling_round = 0
accepted = 0

BATCH_SIZE = 5000
MAX_ROUNDS = 10000


while accepted < TOTAL_TARGET:

    sampling_round += 1

    if sampling_round > MAX_ROUNDS:

        incomplete = {
            cell: len(configs)
            for cell, configs
            in cell_configurations.items()
            if len(configs) < TARGET_PER_CELL
        }

        raise RuntimeError(
            "Maximum sampling rounds reached.\n"
            f"Accepted: {accepted}/{TOTAL_TARGET}\n"
            f"Incomplete cells: {len(incomplete)}\n"
            f"Example incomplete cells: "
            f"{list(incomplete.items())[:10]}"
        )

    # --------------------------------------------------------
    # SAMPLE COMPLETE NATIVE CONFIGURATIONS
    # --------------------------------------------------------

    configs = full_cs.sample_configuration(
        size=BATCH_SIZE
    )

    if not isinstance(configs, list):
        configs = [configs]

    # --------------------------------------------------------
    # PROCESS CONFIGURATIONS
    # --------------------------------------------------------

    for config in configs:

        # ----------------------------------------------------
        # FIRST: REQUIRE PINNED DATA PREPROCESSING
        # ----------------------------------------------------

        if not matches_pinned_preprocessing(config):
            continue

        # ----------------------------------------------------
        # SECOND: IDENTIFY FACTORIAL CELL
        # ----------------------------------------------------

        cell_key = get_factorial_cell(config)

        if cell_key is None:
            continue

        # ----------------------------------------------------
        # CELL ALREADY FULL?
        # ----------------------------------------------------

        if (
            len(cell_configurations[cell_key])
            >= TARGET_PER_CELL
        ):
            continue

        # ----------------------------------------------------
        # DUPLICATE CHECK
        # ----------------------------------------------------

        signature = tuple(
            sorted(
                config.get_dictionary().items()
            )
        )

        if (
            signature
            in cell_configuration_signatures[cell_key]
        ):
            continue

        # ----------------------------------------------------
        # ACCEPT
        # ----------------------------------------------------

        cell_configurations[cell_key].append(
            config
        )

        cell_configuration_signatures[
            cell_key
        ].add(signature)

        accepted += 1

        if accepted >= TOTAL_TARGET:
            break

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (
        sampling_round == 1
        or sampling_round % 10 == 0
        or accepted >= TOTAL_TARGET
    ):

        complete_cells = sum(
            len(configs) >= TARGET_PER_CELL
            for configs
            in cell_configurations.values()
        )

        print(
            f"Round {sampling_round:5d} | "
            f"Accepted {accepted:5d}/"
            f"{TOTAL_TARGET} | "
            f"Cells complete "
            f"{complete_cells:3d}/"
            f"{len(VALID_CELL_SET)}"
        )


# ============================================================
# BUILD CONFIGURATION DATAFRAME
# ============================================================

sampled_configurations = []

config_counter = 0

for cell_key in sorted(VALID_CELL_SET):

    classifier, preprocessor, balancing = cell_key

    configs = cell_configurations[cell_key]

    assert len(configs) == TARGET_PER_CELL

    for replicate_index, config in enumerate(configs):

        config_id = (
            f"config_{config_counter:05d}"
        )

        row = {
            "config_id": config_id,

            "cell_id":
                f"{classifier}__"
                f"{preprocessor}__"
                f"{balancing}",

            "classifier":
                classifier,

            "feature_preprocessor":
                preprocessor,

            "balancing":
                balancing,

            "config_replicate":
                replicate_index,
        }

        # Store the configuration values for inspection.
        #
        # IMPORTANT:
        # The actual ConfigSpace object remains in
        # cell_configurations and will be used for training.
        row.update(
            config.get_dictionary()
        )

        sampled_configurations.append(row)

        config_counter += 1


configurations_df = pd.DataFrame(
    sampled_configurations
)


# ============================================================
# CREATE CONFIG_ID → ORIGINAL CONFIGSPACE OBJECT MAP
# ============================================================

CONFIG_OBJECTS = {}

config_counter = 0

for cell_key in sorted(VALID_CELL_SET):

    for config in cell_configurations[cell_key]:

        config_id = (
            f"config_{config_counter:05d}"
        )

        CONFIG_OBJECTS[config_id] = config

        config_counter += 1


# ============================================================
# VERIFY
# ============================================================

cell_counts = (
    configurations_df
    .groupby(
        [
            "classifier",
            "feature_preprocessor",
            "balancing"
        ]
    )
    .size()
    .rename("n_configurations")
    .reset_index()
)


verification_df = (
    valid_cells_df
    .merge(
        cell_counts,
        on=[
            "classifier",
            "feature_preprocessor",
            "balancing"
        ],
        how="left"
    )
)

verification_df["n_configurations"] = (
    verification_df["n_configurations"]
    .fillna(0)
    .astype(int)
)


missing_cells = verification_df[
    verification_df["n_configurations"] == 0
]

incorrect_cells = verification_df[
    verification_df["n_configurations"]
    != TARGET_PER_CELL
]


# ============================================================
# VERIFY PINNED PREPROCESSING
# ============================================================

pin_check_results = []

for config_id, config in CONFIG_OBJECTS.items():

    values = config.get_dictionary()

    pin_check_results.append(
        all(
            values.get(hp_name) == expected_value
            for hp_name, expected_value
            in PINNED_DATA_PREPROCESSOR.items()
        )
    )

all_pinned = all(pin_check_results)


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURATION POPULATION RESULT")
print("=" * 70)

print(
    f"Theoretical cells       : "
    f"{len(CLASSIFIERS) * len(FEATURE_PREPROCESSORS) * len(BALANCING_STRATEGIES)}"
)

print(
    f"Valid factorial cells   : "
    f"{len(VALID_CELL_SET)}"
)

print(
    f"Configurations per cell : "
    f"{TARGET_PER_CELL}"
)

print(
    f"Expected configurations : "
    f"{TOTAL_TARGET}"
)

print(
    f"Actual configurations   : "
    f"{len(configurations_df)}"
)

print(
    f"Missing cells           : "
    f"{len(missing_cells)}"
)

print(
    f"Incorrect cell counts   : "
    f"{len(incorrect_cells)}"
)

print(
    f"All preprocessing pinned: "
    f"{all_pinned}"
)


# ============================================================
# DISTRIBUTION
# ============================================================

print("\nConfiguration count distribution:")

display(
    cell_counts[
        "n_configurations"
    ]
    .value_counts()
    .sort_index()
    .rename("number_of_cells")
    .to_frame()
)


# ============================================================
# ASSERTIONS
# ============================================================

assert len(configurations_df) == TOTAL_TARGET

assert len(CONFIG_OBJECTS) == TOTAL_TARGET

assert len(missing_cells) == 0

assert len(incorrect_cells) == 0

assert all_pinned

print("\n✓ 330 valid factorial cells populated.")
print("✓ Exactly 5 configurations per cell.")
print("✓ 1,650 configurations generated.")
print("✓ All configurations use native ConfigSpace objects.")
print("✓ Data preprocessing settings are pinned.")
print("✓ No artificial ConfigSpace constraints were added.")

REGENERATING CONTROLLED CONFIGURATION POPULATION
Valid factorial cells : 330
Target per cell       : 5
Total target          : 1650

Pinned data preprocessing:
  data_preprocessor:__choice__ = feature_type
  data_preprocessor:feature_type:categorical_transformer:categorical_encoding:__choice__ = one_hot_encoding
  data_preprocessor:feature_type:categorical_transformer:category_coalescence:__choice__ = minority_coalescer
  data_preprocessor:feature_type:numerical_transformer:imputation:strategy = mean
  data_preprocessor:feature_type:numerical_transformer:rescaling:__choice__ = standardize
Round     1 | Accepted    43/1650 | Cells complete   0/330
Round    10 | Accepted   376/1650 | Cells complete   3/330
Round    20 | Accepted   729/1650 | Cells complete  33/330
Round    30 | Accepted  1073/1650 | Cells complete 101/330
Round    40 | Accepted  1281/1650 | Cells complete 182/330
Round    50 | Accepted  1396/1650 | Cells complete 233/330
Round    60 | Accepted  1466/1650 | Cells complete

,number_of_cells
n_configurations,
5,330



✓ 330 valid factorial cells populated.
✓ Exactly 5 configurations per cell.
✓ 1,650 configurations generated.
✓ All configurations use native ConfigSpace objects.
✓ Data preprocessing settings are pinned.
✓ No artificial ConfigSpace constraints were added.


In [9]:
# ============================================================
# CELL 9 — LOAD AND STANDARDIZE EXPERIMENT DATASETS
# ============================================================

from sklearn.datasets import fetch_openml
from pathlib import Path
import pickle
import time


# ------------------------------------------------------------
# DATASET METADATA
# ------------------------------------------------------------

DATASET_META = {
    37: {
        "name": "diabetes",
    },
    31: {
        "name": "credit-g",
    },
    44: {
        "name": "spambase",
    },
    4534: {
        "name": "phishing_websites",
    },
    1461: {
        "name": "bank_marketing",
    },
}


# ------------------------------------------------------------
# CACHE
# ------------------------------------------------------------

DATA_CACHE_DIR = RESULTS_DIR / "datasets"
DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# TARGET STANDARDIZATION
# ------------------------------------------------------------

def standardize_binary_target(y):
    """
    Convert a binary OpenML target into integer labels {0, 1}
    while preserving the original class mapping.
    """

    y = pd.Series(y).copy()

    # Remove missing target rows
    valid_mask = ~y.isna()
    y = y.loc[valid_mask]

    classes = list(pd.unique(y))

    if len(classes) != 2:
        raise ValueError(
            f"Expected binary target, found {len(classes)} classes: "
            f"{classes}"
        )

    # Deterministic mapping
    classes_sorted = sorted(
        classes,
        key=lambda x: str(x)
    )

    mapping = {
        classes_sorted[0]: 0,
        classes_sorted[1]: 1,
    }

    y_binary = y.map(mapping).astype(int)

    return y_binary, mapping, valid_mask


# ------------------------------------------------------------
# FEATURE TYPE DETECTION
# ------------------------------------------------------------

def detect_feature_types(X):
    """
    Detect Auto-sklearn feature types.

    Numerical columns -> 'numerical'
    Object/category/bool columns -> 'categorical'
    """

    feat_type = []

    for dtype in X.dtypes:

        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_categorical_dtype(dtype)
            or pd.api.types.is_bool_dtype(dtype)
        ):
            feat_type.append("categorical")

        else:
            feat_type.append("numerical")

    return feat_type


# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

DATASETS = {}

for dataset_id in DATASET_IDS:

    print("\n" + "=" * 70)
    print(
        f"LOADING DATASET {dataset_id} — "
        f"{DATASET_META[dataset_id]['name']}"
    )
    print("=" * 70)

    start_time = time.time()

    try:

        # ----------------------------------------------------
        # OpenML
        # ----------------------------------------------------

        try:
            openml_data = fetch_openml(
                data_id=dataset_id,
                as_frame=True,
                parser="auto",
            )

        except TypeError:
            # Compatibility with older scikit-learn
            openml_data = fetch_openml(
                data_id=dataset_id,
                as_frame=True,
            )

        X = openml_data.data.copy()
        y_original = pd.Series(
            openml_data.target,
            name="target"
        )

        # ----------------------------------------------------
        # TARGET
        # ----------------------------------------------------

        y, target_mapping, valid_target_mask = (
            standardize_binary_target(
                y_original
            )
        )

        # Keep X aligned with target after removing missing
        # target observations.
        X = X.loc[valid_target_mask].copy()

        # Reset indices so X/y align exactly.
        X.reset_index(drop=True, inplace=True)
        y.reset_index(drop=True, inplace=True)

        # ----------------------------------------------------
        # FEATURE TYPES
        # ----------------------------------------------------

        feat_type = detect_feature_types(X)

        # ----------------------------------------------------
        # BASIC DATASET INFORMATION
        # ----------------------------------------------------

        n_samples = X.shape[0]
        n_features = X.shape[1]

        n_numeric = sum(
            t == "numerical"
            for t in feat_type
        )

        n_categorical = sum(
            t == "categorical"
            for t in feat_type
        )

        class_counts = (
            y.value_counts()
            .sort_index()
            .to_dict()
        )

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        DATASETS[dataset_id] = {
            "id": dataset_id,
            "name": DATASET_META[dataset_id]["name"],
            "X": X,
            "y": y,
            "feat_type": feat_type,
            "target_mapping": target_mapping,
            "feature_names": list(X.columns),
            "n_samples": n_samples,
            "n_features": n_features,
            "n_numeric": n_numeric,
            "n_categorical": n_categorical,
            "class_counts": class_counts,
        }

        elapsed = time.time() - start_time

        print(f"Name              : {DATASETS[dataset_id]['name']}")
        print(f"Samples           : {n_samples}")
        print(f"Features          : {n_features}")
        print(f"Numerical         : {n_numeric}")
        print(f"Categorical       : {n_categorical}")
        print(f"Class counts      : {class_counts}")
        print(f"Target mapping    : {target_mapping}")
        print(f"Loading time      : {elapsed:.2f} sec")

    except Exception as e:

        print(
            f"\nERROR loading dataset {dataset_id}:"
        )
        print(repr(e))
        raise


# ============================================================
# DATASET SUMMARY TABLE
# ============================================================

dataset_summary = pd.DataFrame([
    {
        "dataset_id": dataset_id,
        "dataset": DATASETS[dataset_id]["name"],
        "n_samples": DATASETS[dataset_id]["n_samples"],
        "n_features": DATASETS[dataset_id]["n_features"],
        "numeric": DATASETS[dataset_id]["n_numeric"],
        "categorical": DATASETS[dataset_id]["n_categorical"],
        "class_0": DATASETS[dataset_id]["class_counts"].get(0, 0),
        "class_1": DATASETS[dataset_id]["class_counts"].get(1, 0),
    }
    for dataset_id in DATASET_IDS
])


print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

display(dataset_summary)


# ============================================================
# SANITY CHECKS
# ============================================================

for dataset_id in DATASET_IDS:

    data = DATASETS[dataset_id]

    assert len(data["X"]) == len(data["y"])

    assert data["y"].isin([0, 1]).all()

    assert len(data["feat_type"]) == data["n_features"]

    assert data["n_samples"] > 0
    assert data["n_features"] > 0


print("\n✓ All five datasets loaded successfully.")
print("✓ All targets are binary {0, 1}.")
print("✓ X/y alignment verified.")
print("✓ Feature-type vectors verified.")


LOADING DATASET 37 — diabetes
Name              : diabetes
Samples           : 768
Features          : 8
Numerical         : 8
Categorical       : 0
Class counts      : {0: 500, 1: 268}
Target mapping    : {'tested_negative': 0, 'tested_positive': 1}
Loading time      : 0.01 sec

LOADING DATASET 31 — credit-g
Name              : credit-g
Samples           : 1000
Features          : 20
Numerical         : 7
Categorical       : 13
Class counts      : {0: 300, 1: 700}
Target mapping    : {'bad': 0, 'good': 1}
Loading time      : 0.03 sec

LOADING DATASET 44 — spambase
Name              : spambase
Samples           : 4601
Features          : 57
Numerical         : 57
Categorical       : 0
Class counts      : {0: 2788, 1: 1813}
Target mapping    : {'0': 0, '1': 1}
Loading time      : 0.09 sec

LOADING DATASET 4534 — phishing_websites
Name              : phishing_websites
Samples           : 11055
Features          : 30
Numerical         : 0
Categorical       : 30
Class counts      : {0: 48

,dataset_id,dataset,n_samples,n_features,numeric,categorical,class_0,class_1
0,37,diabetes,768,8,8,0,500,268
1,31,credit-g,1000,20,7,13,300,700
2,44,spambase,4601,57,57,0,2788,1813
3,4534,phishing_websites,11055,30,0,30,4898,6157
4,1461,bank_marketing,45211,16,7,9,39922,5289



✓ All five datasets loaded successfully.
✓ All targets are binary {0, 1}.
✓ X/y alignment verified.
✓ Feature-type vectors verified.


In [10]:
# ============================================================
# CELL 10 — FIXED STRATIFIED 5-FOLD CV SPLITS
# ============================================================

from sklearn.model_selection import StratifiedKFold


# ------------------------------------------------------------
# CV CONFIGURATION
# ------------------------------------------------------------

CV_SPLITS = {}


# ------------------------------------------------------------
# CREATE IDENTICAL FOLD STRUCTURE FOR ALL CONFIGURATIONS
# ------------------------------------------------------------

for dataset_id in DATASET_IDS:

    data = DATASETS[dataset_id]

    X = data["X"]
    y = data["y"]

    skf = StratifiedKFold(
        n_splits=N_CV_FOLDS,
        shuffle=True,
        random_state=GLOBAL_SEED,
    )

    folds = []

    for fold_id, (train_idx, test_idx) in enumerate(
        skf.split(X, y)
    ):

        folds.append({
            "fold": fold_id,
            "train_idx": train_idx,
            "test_idx": test_idx,
        })

    CV_SPLITS[dataset_id] = folds


# ============================================================
# VERIFY FOLDS
# ============================================================

print("=" * 70)
print("FIXED STRATIFIED CROSS-VALIDATION SPLITS")
print("=" * 70)

for dataset_id in DATASET_IDS:

    data = DATASETS[dataset_id]
    folds = CV_SPLITS[dataset_id]

    print(
        f"\nDataset {dataset_id} — "
        f"{data['name']}"
    )

    print(
        f"Samples: {data['n_samples']} | "
        f"Folds: {len(folds)}"
    )

    for fold in folds:

        train_idx = fold["train_idx"]
        test_idx = fold["test_idx"]

        y_train = data["y"].iloc[train_idx]
        y_test = data["y"].iloc[test_idx]

        print(
            f"  Fold {fold['fold']}: "
            f"train={len(train_idx):5d}, "
            f"test={len(test_idx):5d}, "
            f"train classes="
            f"{y_train.value_counts().sort_index().to_dict()}, "
            f"test classes="
            f"{y_test.value_counts().sort_index().to_dict()}"
        )


# ============================================================
# GLOBAL SANITY CHECKS
# ============================================================

for dataset_id in DATASET_IDS:

    data = DATASETS[dataset_id]
    folds = CV_SPLITS[dataset_id]

    assert len(folds) == N_CV_FOLDS

    all_test_indices = []

    for fold in folds:

        train_idx = fold["train_idx"]
        test_idx = fold["test_idx"]

        # No overlap between train and test
        assert len(
            set(train_idx).intersection(
                set(test_idx)
            )
        ) == 0

        all_test_indices.extend(
            test_idx.tolist()
        )

    # Every sample must appear exactly once as a test sample
    counts = pd.Series(
        all_test_indices
    ).value_counts()

    assert len(counts) == data["n_samples"]
    assert counts.eq(1).all()


print("\n✓ 5 fixed stratified folds created for every dataset.")
print("✓ No train/test overlap within any fold.")
print("✓ Every sample appears exactly once in the test folds.")
print("✓ These folds will be reused for every configuration.")

FIXED STRATIFIED CROSS-VALIDATION SPLITS

Dataset 37 — diabetes
Samples: 768 | Folds: 5
  Fold 0: train=  614, test=  154, train classes={0: 400, 1: 214}, test classes={0: 100, 1: 54}
  Fold 1: train=  614, test=  154, train classes={0: 400, 1: 214}, test classes={0: 100, 1: 54}
  Fold 2: train=  614, test=  154, train classes={0: 400, 1: 214}, test classes={0: 100, 1: 54}
  Fold 3: train=  615, test=  153, train classes={0: 400, 1: 215}, test classes={0: 100, 1: 53}
  Fold 4: train=  615, test=  153, train classes={0: 400, 1: 215}, test classes={0: 100, 1: 53}

Dataset 31 — credit-g
Samples: 1000 | Folds: 5
  Fold 0: train=  800, test=  200, train classes={0: 240, 1: 560}, test classes={0: 60, 1: 140}
  Fold 1: train=  800, test=  200, train classes={0: 240, 1: 560}, test classes={0: 60, 1: 140}
  Fold 2: train=  800, test=  200, train classes={0: 240, 1: 560}, test classes={0: 60, 1: 140}
  Fold 3: train=  800, test=  200, train classes={0: 240, 1: 560}, test classes={0: 60, 1: 140}


In [11]:
# # ============================================================
# # CELL 11 — FULL CONTROLLED CV EXPERIMENT
# #
# # FINAL FIXED VERSION
# #
# # FIXES:
# #   1. Correct Auto-sklearn instance API
# #   2. feat_type list -> {column_index: feature_type}
# #   3. Integer HP float -> integer conversion
# #   4. Dataset-inapplicable conditional HPs are removed
# #      when constructing dataset-specific configurations
# #   5. Thread oversubscription
# #   6. Resume existing checkpoints
# #   7. Remove previously-created bogus UniformInteger errors
# #   8. No timeout
# #   9. No arbitrary configuration skipping
# #  10. Immediate checkpointing
# # ============================================================


# # ============================================================
# # IMPORTS
# # ============================================================

# import time
# import traceback

# import numpy as np
# import pandas as pd

# from sklearn.metrics import log_loss

# from threadpoolctl import threadpool_limits

# from ConfigSpace import Configuration

# from autosklearn.pipeline.classification import (
#     SimpleClassificationPipeline
# )


# # ============================================================
# # SETTINGS
# # ============================================================

# TRAINING_SEEDS = [
#     GLOBAL_SEED + i
#     for i in range(N_TRAINING_SEEDS)
# ]

# THREAD_LIMIT = 1


# # ============================================================
# # RESULT DIRECTORIES
# # ============================================================

# CV_RESULT_DIR = (
#     RESULTS_DIR / "cv_results"
# )

# CV_RESULT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# CV_ERROR_DIR = (
#     CV_RESULT_DIR / "errors"
# )

# CV_ERROR_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )


# # ============================================================
# # REBUILD ORIGINAL CONFIG OBJECTS
# #
# # IMPORTANT:
# # These are the ORIGINAL native ConfigSpace
# # configurations sampled earlier.
# # ============================================================

# CONFIG_OBJECTS = {}

# config_counter = 0


# for cell_key in sorted(
#     VALID_CELL_SET
# ):

#     configs = (
#         cell_configurations[
#             cell_key
#         ]
#     )

#     for config in configs:

#         config_id = (
#             f"config_{config_counter:05d}"
#         )

#         CONFIG_OBJECTS[
#             config_id
#         ] = config

#         config_counter += 1


# print("=" * 80)
# print("CONFIGURATION OBJECT VERIFICATION")
# print("=" * 80)

# print(
#     "Configuration objects:",
#     len(CONFIG_OBJECTS)
# )

# print(
#     "Expected:",
#     len(configurations_df)
# )

# assert (
#     len(CONFIG_OBJECTS)
#     ==
#     len(configurations_df)
# )

# print(
#     "✓ Original native ConfigSpace objects recovered."
# )


# # ============================================================
# # FEATURE TYPE NORMALIZATION
# #
# # Auto-sklearn expects:
# #
# # {
# #     0: "numerical",
# #     1: "categorical",
# #     ...
# # }
# #
# # Your dataset definitions may contain:
# #
# # [
# #     "numerical",
# #     "categorical",
# #     ...
# # ]
# # ============================================================

# def normalize_feat_type(
#     feat_type
# ):

#     # --------------------------------------------------------
#     # Already dictionary
#     # --------------------------------------------------------

#     if isinstance(
#         feat_type,
#         dict
#     ):

#         return {
#             int(k): v
#             for k, v in feat_type.items()
#         }


#     # --------------------------------------------------------
#     # List / tuple / numpy array
#     # --------------------------------------------------------

#     if isinstance(
#         feat_type,
#         (
#             list,
#             tuple,
#             np.ndarray,
#         )
#     ):

#         return {
#             int(i): value
#             for i, value
#             in enumerate(feat_type)
#         }


#     raise TypeError(
#         "Unsupported feat_type type: "
#         f"{type(feat_type)}"
#     )


# # ============================================================
# # DATASET PROPERTIES
# # ============================================================

# def make_dataset_properties(
#     dataset_id
# ):

#     data = DATASETS[
#         dataset_id
#     ]

#     feat_type = normalize_feat_type(
#         data["feat_type"]
#     )

#     return {
#         "multilabel": False,
#         "multiclass": False,
#         "binary": True,
#         "sparse": False,
#         "signed": False,
#         "input": feat_type,
#         "output": ["binary"],
#     }


# # ============================================================
# # PIPELINE CONFIGSPACE CACHE
# # ============================================================

# PIPELINE_CONFIG_SPACES = {}


# # ============================================================
# # GET DATASET-SPECIFIC AUTOSKLEARN CONFIGSPACE
# # ============================================================

# def get_pipeline_config_space(
#     dataset_id,
#     dataset_properties,
# ):

#     if (
#         dataset_id
#         in
#         PIPELINE_CONFIG_SPACES
#     ):

#         return (
#             PIPELINE_CONFIG_SPACES[
#                 dataset_id
#             ]
#         )


#     # --------------------------------------------------------
#     # Make a copy because Auto-sklearn modifies
#     # dataset_properties internally.
#     # --------------------------------------------------------

#     local_properties = dict(
#         dataset_properties
#     )


#     # --------------------------------------------------------
#     # Create pipeline instance.
#     #
#     # IMPORTANT:
#     # get_hyperparameter_search_space() is an INSTANCE method
#     # in Auto-sklearn 0.15.0.
#     # --------------------------------------------------------

#     pipeline = (
#         SimpleClassificationPipeline(
#             config=None,
#             feat_type=
#                 local_properties[
#                     "input"
#                 ],
#             dataset_properties=
#                 local_properties,
#             include=None,
#             exclude=None,
#             random_state=
#                 GLOBAL_SEED,
#         )
#     )


#     config_space = (
#         pipeline
#         .get_hyperparameter_search_space(
#             feat_type=
#                 local_properties[
#                     "input"
#                 ],
#             dataset_properties=
#                 local_properties,
#         )
#     )


#     PIPELINE_CONFIG_SPACES[
#         dataset_id
#     ] = config_space


#     return config_space


# # ============================================================
# # CONFIGURATION NORMALIZATION
# #
# # PURPOSE:
# #
# # Convert the ORIGINAL native configuration into a
# # dataset-specific SimpleClassificationPipeline configuration.
# #
# # We:
# #
# #   - preserve all applicable values
# #   - convert integer values correctly
# #   - convert float values correctly
# #   - remove only hyperparameters that do not exist in the
# #     dataset-specific conditional space
# #
# # We DO NOT change:
# #
# #   classifier family
# #   feature preprocessor
# #   balancing
# #   applicable hyperparameter values
# # ============================================================

# def normalize_configuration_types(
#     config,
#     dataset_id,
#     dataset_properties,
# ):

#     target_cs = (
#         get_pipeline_config_space(
#             dataset_id=
#                 dataset_id,
#             dataset_properties=
#                 dataset_properties,
#         )
#     )


#     original_values = (
#         config.get_dictionary()
#     )


#     normalized_values = {}

#     omitted_parameters = []


#     # ========================================================
#     # ITERATE THROUGH ORIGINAL CONFIGURATION
#     # ========================================================

#     for (
#         name,
#         value
#     ) in original_values.items():


#         # ====================================================
#         # HYPERPARAMETER NOT AVAILABLE FOR THIS DATASET
#         # ====================================================

#         if name not in target_cs:

#             omitted_parameters.append(
#                 name
#             )

#             continue


#         hp = target_cs[
#             name
#         ]


#         hp_class = (
#             hp.__class__.__name__
#         )


#         # ====================================================
#         # INTEGER HYPERPARAMETER
#         # ====================================================

#         if hp_class.endswith(
#             "IntegerHyperparameter"
#         ):

#             # ------------------------------------------------
#             # Already integer
#             # ------------------------------------------------

#             if isinstance(
#                 value,
#                 (
#                     int,
#                     np.integer,
#                 )
#             ):

#                 normalized_values[
#                     name
#                 ] = int(value)


#             # ------------------------------------------------
#             # Float representation of integer
#             # ------------------------------------------------

#             elif isinstance(
#                 value,
#                 (
#                     float,
#                     np.floating,
#                 )
#             ):

#                 if not np.isfinite(
#                     value
#                 ):

#                     raise ValueError(
#                         f"Non-finite integer "
#                         f"value for "
#                         f"'{name}': {value}"
#                     )


#                 if float(
#                     value
#                 ).is_integer():

#                     normalized_values[
#                         name
#                     ] = int(value)

#                 else:

#                     raise ValueError(
#                         f"Cannot convert "
#                         f"non-integral value "
#                         f"{value} to integer "
#                         f"hyperparameter "
#                         f"'{name}'."
#                     )


#             else:

#                 raise TypeError(
#                     f"Unexpected value type "
#                     f"for integer HP "
#                     f"'{name}': "
#                     f"{type(value)}"
#                 )


#         # ====================================================
#         # FLOAT HYPERPARAMETER
#         # ====================================================

#         elif hp_class.endswith(
#             "FloatHyperparameter"
#         ):

#             normalized_values[
#                 name
#             ] = float(value)


#         # ====================================================
#         # EVERYTHING ELSE
#         # ====================================================

#         else:

#             normalized_values[
#                 name
#             ] = value


#     # ========================================================
#     # BUILD CONFIGURATION
#     # ========================================================

#     try:

#         normalized_config = (
#             Configuration(
#                 configuration_space=
#                     target_cs,
#                 values=
#                     normalized_values,
#             )
#         )


#     except Exception as e:

#         print()
#         print("=" * 80)
#         print(
#             "DATASET-SPECIFIC CONFIGURATION "
#             "CONSTRUCTION FAILED"
#         )
#         print("=" * 80)

#         print(
#             "Dataset:",
#             dataset_id
#         )

#         print(
#             "Exception:",
#             type(e).__name__
#         )

#         print(
#             "Message:",
#             str(e)
#         )

#         print()
#         print(
#             "Original configuration:"
#         )

#         print(
#             config
#         )

#         print()
#         print(
#             "Normalized values:"
#         )

#         print(
#             normalized_values
#         )

#         print()
#         print(
#             "Omitted parameters:"
#         )

#         for name in (
#             omitted_parameters
#         ):

#             print(
#                 "   ",
#                 name
#             )

#         print(
#             "=" * 80
#         )

#         raise


#     # ========================================================
#     # RETURN
#     # ========================================================

#     return (
#         normalized_config,
#         omitted_parameters,
#     )


# # ============================================================
# # BUILD PIPELINE
# # ============================================================

# def build_pipeline(
#     config,
#     dataset_id,
#     dataset_properties,
#     seed,
# ):

#     normalized_config, _ = (
#         normalize_configuration_types(
#             config=config,
#             dataset_id=
#                 dataset_id,
#             dataset_properties=
#                 dataset_properties,
#         )
#     )


#     local_properties = dict(
#         dataset_properties
#     )


#     pipeline = (
#         SimpleClassificationPipeline(
#             config=
#                 normalized_config,
#             feat_type=
#                 local_properties[
#                     "input"
#                 ],
#             dataset_properties=
#                 local_properties,
#             include=None,
#             exclude=None,
#             random_state=
#                 seed,
#         )
#     )


#     return pipeline


# # ============================================================
# # SINGLE FOLD EVALUATION
# # ============================================================

# def evaluate_fold(
#     config,
#     dataset_id,
#     dataset_properties,
#     X,
#     y,
#     train_idx,
#     test_idx,
#     seed,
# ):

#     X_train = X.iloc[
#         train_idx
#     ]

#     X_test = X.iloc[
#         test_idx
#     ]

#     y_train = y.iloc[
#         train_idx
#     ]

#     y_test = y.iloc[
#         test_idx
#     ]


#     # ========================================================
#     # LIMIT ALL NATIVE THREAD POOLS
#     #
#     # This is essential because the unrestricted version
#     # previously created hundreds of threads.
#     # ========================================================

#     with threadpool_limits(
#         limits=THREAD_LIMIT
#     ):

#         # ----------------------------------------------------
#         # BUILD
#         # ----------------------------------------------------

#         pipeline = build_pipeline(
#             config=config,
#             dataset_id=
#                 dataset_id,
#             dataset_properties=
#                 dataset_properties,
#             seed=seed,
#         )


#         # ----------------------------------------------------
#         # FIT
#         # ----------------------------------------------------

#         pipeline.fit(
#             X_train,
#             y_train,
#         )


#         # ----------------------------------------------------
#         # PREDICT
#         # ----------------------------------------------------

#         probabilities = (
#             pipeline.predict_proba(
#                 X_test
#             )
#         )


#         probabilities = np.asarray(
#             probabilities
#         )


#         # ----------------------------------------------------
#         # POSITIVE CLASS
#         # ----------------------------------------------------

#         positive_probability = (
#             probabilities[:, 1]
#         )


#         positive_probability = (
#             np.clip(
#                 positive_probability,
#                 1e-15,
#                 1 - 1e-15,
#             )
#         )


#     # ========================================================
#     # LOG LOSS
#     # ========================================================

#     fold_log_loss = (
#         log_loss(
#             y_test,
#             positive_probability,
#             labels=[0, 1],
#         )
#     )


#     return float(
#         fold_log_loss
#     )


# # ============================================================
# # CHECKPOINT LOADER
# # ============================================================

# def load_checkpoint(
#     path
# ):

#     if not path.exists():

#         return pd.DataFrame()


#     try:

#         return pd.read_csv(
#             path
#         )

#     except Exception as e:

#         print(
#             f"WARNING: Could not read "
#             f"{path}: "
#             f"{type(e).__name__}: {e}"
#         )

#         return pd.DataFrame()


# # ============================================================
# # EXECUTION KEY
# # ============================================================

# def make_execution_key(
#     config_id,
#     training_seed,
#     fold_id,
# ):

#     return (
#         str(config_id),
#         int(training_seed),
#         int(fold_id),
#     )


# # ============================================================
# # REMOVE KNOWN BOGUS UniformInteger ERRORS
# #
# # These were created by the earlier implementation which
# # passed integer-valued floats such as 4.0 into ConfigSpace.
# #
# # Only remove those specific errors.
# # ============================================================

# BOGUS_DATASETS = [
#     31,
#     44,
#     4534,
#     1461,
# ]


# def remove_bogus_type_errors(
#     dataset_id
# ):

#     error_path = (
#         CV_ERROR_DIR
#         /
#         f"errors_{dataset_id}.csv"
#     )


#     if not error_path.exists():

#         print(
#             f"[{dataset_id}] "
#             f"No error checkpoint."
#         )

#         return


#     errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     if errors.empty:

#         print(
#             f"[{dataset_id}] "
#             f"Error checkpoint empty."
#         )

#         return


#     error_text = (
#         errors[
#             "error"
#         ]
#         .fillna("")
#         .astype(str)
#     )


#     bogus_mask = (
#         error_text.str.contains(
#             "Trying to set illegal value",
#             regex=False,
#         )
#         &
#         error_text.str.contains(
#             "UniformInteger",
#             regex=False,
#         )
#     )


#     bogus_count = int(
#         bogus_mask.sum()
#     )


#     if bogus_count == 0:

#         print(
#             f"[{dataset_id}] "
#             f"No bogus UniformInteger errors."
#         )

#         return


#     cleaned_errors = (
#         errors.loc[
#             ~bogus_mask
#         ]
#         .copy()
#     )


#     if cleaned_errors.empty:

#         error_path.unlink()

#         print(
#             f"[{dataset_id}] "
#             f"Removed {bogus_count} "
#             f"bogus errors."
#         )

#     else:

#         cleaned_errors.to_csv(
#             error_path,
#             index=False,
#         )

#         print(
#             f"[{dataset_id}] "
#             f"Removed {bogus_count} "
#             f"bogus errors; "
#             f"preserved "
#             f"{len(cleaned_errors)} "
#             f"genuine errors."
#         )


# # ============================================================
# # CLEAN OLD BOGUS ERRORS
# # ============================================================

# print()
# print("=" * 80)
# print(
#     "CLEANING PREVIOUS BOGUS CONFIGSPACE ERRORS"
# )
# print("=" * 80)


# for dataset_id in (
#     BOGUS_DATASETS
# ):

#     remove_bogus_type_errors(
#         dataset_id
#     )


# # ============================================================
# # PRECHECK DATASET 31
# #
# # We test the exact configuration that previously failed.
# # ============================================================

# print()
# print("=" * 80)
# print(
#     "PRECHECK — DATASET 31 / CONFIG_00000"
# )
# print("=" * 80)


# test_dataset_id = 31


# test_dataset_properties = (
#     make_dataset_properties(
#         test_dataset_id
#     )
# )


# print(
#     "feat_type type:",
#     type(
#         test_dataset_properties[
#             "input"
#         ]
#     )
# )


# print(
#     "feat_type:",
#     test_dataset_properties[
#         "input"
#     ]
# )


# assert isinstance(
#     test_dataset_properties[
#         "input"
#     ],
#     dict
# )


# test_original_config = (
#     CONFIG_OBJECTS[
#         "config_00000"
#     ]
# )


# test_original_values = (
#     test_original_config
#     .get_dictionary()
# )


# (
#     test_normalized_config,
#     test_omitted_parameters,
# ) = (
#     normalize_configuration_types(
#         config=
#             test_original_config,
#         dataset_id=
#             test_dataset_id,
#         dataset_properties=
#             test_dataset_properties,
#     )
# )


# test_normalized_values = (
#     test_normalized_config
#     .get_dictionary()
# )


# print()
# print(
#     "Original hyperparameters:",
#     len(
#         test_original_values
#     )
# )


# print(
#     "Applicable target hyperparameters:",
#     len(
#         test_normalized_values
#     )
# )


# print(
#     "Omitted dataset-inapplicable:",
#     len(
#         test_omitted_parameters
#     )
# )


# # ============================================================
# # SHOW INTEGER CONVERSION
# # ============================================================

# if (
#     "classifier:adaboost:max_depth"
#     in test_original_values
# ):

#     original_max_depth = (
#         test_original_values[
#             "classifier:adaboost:max_depth"
#         ]
#     )

#     normalized_max_depth = (
#         test_normalized_values[
#             "classifier:adaboost:max_depth"
#         ]
#     )


#     print()
#     print(
#         "AdaBoost max_depth:"
#     )

#     print(
#         "  original :",
#         original_max_depth,
#         type(
#             original_max_depth
#         )
#     )

#     print(
#         "  normalized:",
#         normalized_max_depth,
#         type(
#             normalized_max_depth
#         )
#     )


# # ============================================================
# # SHOW OMITTED STRUCTURAL PARAMETERS
# # ============================================================

# if test_omitted_parameters:

#     print()
#     print(
#         "Dataset-inapplicable conditional "
#         "parameters omitted:"
#     )

#     for name in (
#         test_omitted_parameters
#     ):

#         print(
#             "   ",
#             name
#         )


# # ============================================================
# # PRECHECK SUCCESS
# # ============================================================

# print()
# print(
#     "✓ Dataset 31 config_00000 "
#     "successfully mapped to its "
#     "dataset-specific ConfigSpace."
# )


# # ============================================================
# # FULL DATASET EXPERIMENT
# # ============================================================

# ALL_DATASET_RESULTS = {}


# for dataset_id in DATASET_IDS:

#     dataset_name = (
#         DATASETS[
#             dataset_id
#         ]["name"]
#     )


#     print("\n\n")
#     print("=" * 80)
#     print(
#         f"DATASET {dataset_id} — "
#         f"{dataset_name}"
#     )
#     print("=" * 80)


#     # ========================================================
#     # CHECKPOINT PATHS
#     # ========================================================

#     result_path = (
#         CV_RESULT_DIR
#         /
#         f"cv_results_{dataset_id}.csv"
#     )


#     error_path = (
#         CV_ERROR_DIR
#         /
#         f"errors_{dataset_id}.csv"
#     )


#     # ========================================================
#     # LOAD CHECKPOINTS
#     # ========================================================

#     existing_results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     existing_errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     # ========================================================
#     # DATA
#     # ========================================================

#     data = DATASETS[
#         dataset_id
#     ]


#     X = data["X"]

#     y = data["y"]


#     # ========================================================
#     # DATASET PROPERTIES
#     # ========================================================

#     dataset_properties = (
#         make_dataset_properties(
#             dataset_id
#         )
#     )


#     print(
#         "Feature-type representation:",
#         type(
#             dataset_properties[
#                 "input"
#             ]
#         )
#     )


#     # ========================================================
#     # BUILD DATASET-SPECIFIC CONFIGSPACE
#     # ========================================================

#     target_config_space = (
#         get_pipeline_config_space(
#             dataset_id=
#                 dataset_id,
#             dataset_properties=
#                 dataset_properties,
#         )
#     )


#     print(
#         "Target ConfigSpace ready."
#     )


#     print(
#         "Target hyperparameters:",
#         len(
#             target_config_space
#             .get_hyperparameters()
#         )
#     )


#     # ========================================================
#     # FIXED FOLDS
#     # ========================================================

#     folds = (
#         CV_SPLITS[
#             dataset_id
#         ]
#     )


#     assert (
#         len(folds)
#         ==
#         N_CV_FOLDS
#     )


#     # ========================================================
#     # EXPECTED EXECUTIONS
#     #
#     # 1650 configurations
#     # × 3 training seeds
#     # × 5 folds
#     # = 24,750
#     # ========================================================

#     total_folds = (
#         len(
#             configurations_df
#         )
#         *
#         len(
#             TRAINING_SEEDS
#         )
#         *
#         N_CV_FOLDS
#     )


#     # ========================================================
#     # RESUME KEYS
#     # ========================================================

#     completed_keys = set()


#     # --------------------------------------------------------
#     # SUCCESSFUL EXECUTIONS
#     # --------------------------------------------------------

#     if not existing_results.empty:

#         for _, row in (
#             existing_results.iterrows()
#         ):

#             completed_keys.add(
#                 make_execution_key(
#                     row["config_id"],
#                     row["training_seed"],
#                     row["fold"],
#                 )
#             )


#     # --------------------------------------------------------
#     # FAILED EXECUTIONS
#     # --------------------------------------------------------

#     if not existing_errors.empty:

#         for _, row in (
#             existing_errors.iterrows()
#         ):

#             completed_keys.add(
#                 make_execution_key(
#                     row["config_id"],
#                     row["training_seed"],
#                     row["fold"],
#                 )
#             )


#     completed_count = (
#         len(
#             completed_keys
#         )
#     )


#     print()
#     print(
#         "Existing successful:",
#         len(existing_results)
#     )

#     print(
#         "Existing failures:",
#         len(existing_errors)
#     )

#     print(
#         "Unique completed:",
#         completed_count
#     )

#     print(
#         "Expected:",
#         total_folds
#     )

#     print(
#         "Remaining:",
#         total_folds -
#         completed_count
#     )


#     dataset_start = (
#         time.time()
#     )


#     # ========================================================
#     # CONFIGURATION LOOP
#     # ========================================================

#     for (
#         config_number,
#         (_, config_row)
#     ) in enumerate(
#         configurations_df.iterrows(),
#         start=1,
#     ):

#         config_id = (
#             config_row[
#                 "config_id"
#             ]
#         )


#         config = (
#             CONFIG_OBJECTS[
#                 config_id
#             ]
#         )


#         # ====================================================
#         # TRAINING SEEDS
#         # ====================================================

#         for training_seed in (
#             TRAINING_SEEDS
#         ):


#             # =================================================
#             # FIVE FIXED FOLDS
#             # =================================================

#             for fold_info in folds:

#                 fold_id = (
#                     fold_info[
#                         "fold"
#                     ]
#                 )


#                 execution_key = (
#                     make_execution_key(
#                         config_id,
#                         training_seed,
#                         fold_id,
#                     )
#                 )


#                 # =============================================
#                 # RESUME
#                 # =============================================

#                 if (
#                     execution_key
#                     in completed_keys
#                 ):

#                     continue


#                 fold_start = (
#                     time.time()
#                 )


#                 # =============================================
#                 # EXECUTE
#                 # =============================================

#                 try:

#                     fold_log_loss = (
#                         evaluate_fold(
#                             config=
#                                 config,
#                             dataset_id=
#                                 dataset_id,
#                             dataset_properties=
#                                 dataset_properties,
#                             X=X,
#                             y=y,
#                             train_idx=
#                                 fold_info[
#                                     "train_idx"
#                                 ],
#                             test_idx=
#                                 fold_info[
#                                     "test_idx"
#                                 ],
#                             seed=
#                                 training_seed,
#                         )
#                     )


#                     # =========================================
#                     # SUCCESS ROW
#                     # =========================================

#                     result_row = {

#                         "dataset_id":
#                             dataset_id,

#                         "dataset":
#                             dataset_name,

#                         "config_id":
#                             config_id,

#                         "classifier":
#                             config_row[
#                                 "classifier"
#                             ],

#                         "feature_preprocessor":
#                             config_row[
#                                 "feature_preprocessor"
#                             ],

#                         "balancing":
#                             config_row[
#                                 "balancing"
#                             ],

#                         "config_replicate":
#                             config_row[
#                                 "config_replicate"
#                             ],

#                         "training_seed":
#                             training_seed,

#                         "fold":
#                             fold_id,

#                         "log_loss":
#                             fold_log_loss,

#                         "status":
#                             "success",

#                         "runtime_seconds":
#                             (
#                                 time.time()
#                                 -
#                                 fold_start
#                             ),
#                     }


#                     # =========================================
#                     # IMMEDIATE CHECKPOINT
#                     # =========================================

#                     pd.DataFrame(
#                         [result_row]
#                     ).to_csv(
#                         result_path,
#                         mode="a",
#                         header=(
#                             not result_path.exists()
#                         ),
#                         index=False,
#                     )


#                     completed_keys.add(
#                         execution_key
#                     )


#                     completed_count += 1


#                 # =============================================
#                 # FAILURE
#                 # =============================================

#                 except Exception as e:

#                     error_row = {

#                         "dataset_id":
#                             dataset_id,

#                         "dataset":
#                             dataset_name,

#                         "config_id":
#                             config_id,

#                         "classifier":
#                             config_row[
#                                 "classifier"
#                             ],

#                         "feature_preprocessor":
#                             config_row[
#                                 "feature_preprocessor"
#                             ],

#                         "balancing":
#                             config_row[
#                                 "balancing"
#                             ],

#                         "config_replicate":
#                             config_row[
#                                 "config_replicate"
#                             ],

#                         "training_seed":
#                             training_seed,

#                         "fold":
#                             fold_id,

#                         "error_type":
#                             type(e).__name__,

#                         "error":
#                             repr(e),

#                         "runtime_seconds":
#                             (
#                                 time.time()
#                                 -
#                                 fold_start
#                             ),

#                         "status":
#                             "failed",
#                     }


#                     # =========================================
#                     # IMMEDIATE FAILURE CHECKPOINT
#                     # =========================================

#                     pd.DataFrame(
#                         [error_row]
#                     ).to_csv(
#                         error_path,
#                         mode="a",
#                         header=(
#                             not error_path.exists()
#                         ),
#                         index=False,
#                     )


#                     completed_keys.add(
#                         execution_key
#                     )


#                     completed_count += 1


#                     print(
#                         f"[{dataset_id}] "
#                         f"FAIL | "
#                         f"{config_id} | "
#                         f"{config_row['classifier']} | "
#                         f"{config_row['feature_preprocessor']} | "
#                         f"{config_row['balancing']} | "
#                         f"seed={training_seed} | "
#                         f"fold={fold_id} | "
#                         f"{type(e).__name__}: "
#                         f"{str(e)[:180]}"
#                     )


#                 # =============================================
#                 # PROGRESS
#                 # =============================================

#                 if (
#                     completed_count % 25 == 0
#                     or
#                     completed_count ==
#                     total_folds
#                 ):

#                     elapsed = (
#                         time.time()
#                         -
#                         dataset_start
#                     )


#                     rate = (
#                         completed_count
#                         /
#                         max(
#                             elapsed / 60,
#                             1e-9,
#                         )
#                     )


#                     remaining = (
#                         total_folds
#                         -
#                         completed_count
#                     )


#                     eta_minutes = (
#                         remaining / rate
#                         if rate > 0
#                         else np.nan
#                     )


#                     print(
#                         f"[{dataset_id}] "
#                         f"{completed_count}/"
#                         f"{total_folds} "
#                         f"("
#                         f"{100 * completed_count / total_folds:.2f}%"
#                         f") | "
#                         f"config "
#                         f"{config_number}/"
#                         f"{len(configurations_df)} | "
#                         f"elapsed "
#                         f"{elapsed / 60:.1f} min | "
#                         f"ETA "
#                         f"{eta_minutes / 60:.2f} h"
#                     )


#     # ========================================================
#     # FINAL CHECKPOINTS
#     # ========================================================

#     final_results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     final_errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     ALL_DATASET_RESULTS[
#         dataset_id
#     ] = final_results


#     # ========================================================
#     # UNIQUE EXECUTION COUNT
#     #
#     # Use keys rather than simply len() so duplicate checkpoint
#     # rows cannot falsely make a dataset appear complete.
#     # ========================================================

#     final_keys = set()


#     for _, row in (
#         final_results.iterrows()
#     ):

#         final_keys.add(
#             make_execution_key(
#                 row["config_id"],
#                 row["training_seed"],
#                 row["fold"],
#             )
#         )


#     for _, row in (
#         final_errors.iterrows()
#     ):

#         final_keys.add(
#             make_execution_key(
#                 row["config_id"],
#                 row["training_seed"],
#                 row["fold"],
#             )
#         )


#     unique_attempted = (
#         len(final_keys)
#     )


#     successful_count = (
#         len(final_results)
#     )


#     failed_count = (
#         len(final_errors)
#     )


#     # ========================================================
#     # DATASET SUMMARY
#     # ========================================================

#     print()
#     print("-" * 80)

#     print(
#         f"DATASET {dataset_id} COMPLETE"
#     )

#     print("-" * 80)

#     print(
#         "Successful rows:",
#         successful_count
#     )

#     print(
#         "Failure rows:",
#         failed_count
#     )

#     print(
#         "Unique executions:",
#         unique_attempted
#     )

#     print(
#         "Expected executions:",
#         total_folds
#     )


#     # ========================================================
#     # STRICT COMPLETENESS CHECK
#     # ========================================================

#     if (
#         unique_attempted
#         !=
#         total_folds
#     ):

#         missing = (
#             total_folds
#             -
#             unique_attempted
#         )


#         raise RuntimeError(
#             f"DATASET {dataset_id} "
#             f"IS NOT COMPLETE. "
#             f"Missing {missing} "
#             f"unique executions."
#         )


#     print(
#         f"✓ Dataset {dataset_id} "
#         f"has exactly "
#         f"{total_folds} unique executions."
#     )


# # ============================================================
# # GLOBAL SUMMARY
# # ============================================================

# print("\n\n")

# print("=" * 80)

# print(
#     "FULL CV EXPERIMENT COMPLETE"
# )

# print("=" * 80)


# grand_success = 0
# grand_failure = 0
# grand_unique = 0


# for dataset_id in DATASET_IDS:

#     dataset_name = (
#         DATASETS[
#             dataset_id
#         ]["name"]
#     )


#     result_path = (
#         CV_RESULT_DIR
#         /
#         f"cv_results_{dataset_id}.csv"
#     )


#     error_path = (
#         CV_ERROR_DIR
#         /
#         f"errors_{dataset_id}.csv"
#     )


#     results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     success_count = (
#         len(results)
#     )


#     failure_count = (
#         len(errors)
#     )


#     unique_keys = set()


#     for _, row in (
#         results.iterrows()
#     ):

#         unique_keys.add(
#             make_execution_key(
#                 row["config_id"],
#                 row["training_seed"],
#                 row["fold"],
#             )
#         )


#     for _, row in (
#         errors.iterrows()
#     ):

#         unique_keys.add(
#             make_execution_key(
#                 row["config_id"],
#                 row["training_seed"],
#                 row["fold"],
#             )
#         )


#     unique_count = (
#         len(unique_keys)
#     )


#     grand_success += (
#         success_count
#     )

#     grand_failure += (
#         failure_count
#     )

#     grand_unique += (
#         unique_count
#     )


#     print(
#         f"{dataset_id} | "
#         f"{dataset_name:25s} | "
#         f"{success_count:6d} successful | "
#         f"{failure_count:6d} failed | "
#         f"{unique_count:6d} unique"
#     )


# # ============================================================
# # GLOBAL EXPECTED
# # ============================================================

# grand_expected = (
#     len(DATASET_IDS)
#     *
#     len(configurations_df)
#     *
#     len(TRAINING_SEEDS)
#     *
#     N_CV_FOLDS
# )


# print("-" * 80)

# print(
#     "TOTAL SUCCESSFUL:",
#     grand_success
# )

# print(
#     "TOTAL FAILED:",
#     grand_failure
# )

# print(
#     "TOTAL UNIQUE EXECUTIONS:",
#     grand_unique
# )

# print(
#     "TOTAL EXPECTED:",
#     grand_expected
# )

# print("-" * 80)


# # ============================================================
# # FINAL ASSERTION
# # ============================================================

# assert (
#     grand_unique
#     ==
#     grand_expected
# )


# print(
#     "✓ ALL DATASETS HAVE THE EXPECTED "
#     "NUMBER OF UNIQUE EXECUTIONS."
# )

# print(
#     "✓ FULL CV EXPERIMENT FINISHED."
# )

In [12]:
# # ============================================================
# # DIAGNOSE DATASET FAILURES
# # ============================================================

# import pandas as pd
# from pathlib import Path

# for dataset_id in [31, 44, 4534, 1461]:

#     error_path = (
#         RESULTS_DIR
#         / "cv_results"
#         / "errors"
#         / f"errors_{dataset_id}.csv"
#     )

#     print("\n")
#     print("=" * 100)
#     print(f"DATASET {dataset_id}")
#     print("=" * 100)

#     errors = pd.read_csv(error_path)

#     print("Total failures:", len(errors))

#     # --------------------------------------------------------
#     # Error type distribution
#     # --------------------------------------------------------

#     print("\nERROR TYPES:")
#     print(
#         errors["error_type"]
#         .value_counts()
#         .to_string()
#     )

#     # --------------------------------------------------------
#     # First error
#     # --------------------------------------------------------

#     print("\nFIRST ERROR:")
#     print(
#         errors.iloc[0]["error"]
#     )

#     # --------------------------------------------------------
#     # First 5 unique error messages
#     # --------------------------------------------------------

#     print("\nFIRST 5 UNIQUE ERRORS:")

#     unique_errors = (
#         errors["error"]
#         .drop_duplicates()
#         .head(5)
#     )

#     for i, error in enumerate(
#         unique_errors,
#         start=1
#     ):

#         print(f"\n--- ERROR {i} ---")
#         print(error)

#     # --------------------------------------------------------
#     # Which configurations fail?
#     # --------------------------------------------------------

#     print("\nCONFIGURATION FAILURE COUNTS:")

#     config_counts = (
#         errors
#         .groupby(
#             [
#                 "config_id",
#                 "classifier",
#                 "feature_preprocessor",
#                 "balancing",
#             ]
#         )
#         .size()
#         .sort_values(
#             ascending=False
#         )
#         .head(15)
#     )

#     print(
#         config_counts.to_string()
#     )

In [13]:
# # ============================================================
# # CELL 11 — FULL CONTROLLED CV EXPERIMENT
# #
# # FINAL FIXED VERSION
# #
# # PURPOSE
# # -------
# # Run the complete controlled Auto-sklearn CV experiment using
# # the EXISTING native ConfigSpace configurations.
# #
# # IMPORTANT:
# #   1. NO new configurations are sampled here.
# #   2. Existing cell_configurations are reused.
# #   3. Original X column names are converted to integer
# #      positions ONLY for the Auto-sklearn representation.
# #   4. feat_type uses exactly the same integer column positions.
# #   5. Dataset categorical/numerical structure is explicitly
# #      defined for the five datasets.
# #   6. The same transformed X representation is used in:
# #          - ConfigSpace construction
# #          - pipeline construction
# #          - fit
# #          - prediction
# #   7. Integer ConfigSpace values such as 4.0 -> 4 are fixed.
# #   8. Threading is limited to 1 to prevent oversubscription.
# #   9. Existing successful results are resumed.
# #  10. Existing OLD UniformInteger construction failures are
# #      removed; genuine failures are preserved.
# #  11. No arbitrary timeout/skip is introduced.
# # ============================================================


# # ============================================================
# # 0. IMPORTS
# # ============================================================

# import time
# import traceback



# import numpy as np
# import pandas as pd

# from sklearn.metrics import log_loss

# from threadpoolctl import threadpool_limits

# from ConfigSpace import Configuration

# from autosklearn.pipeline.classification import (
#     SimpleClassificationPipeline
# )


# # ============================================================
# # MEMORY CLEANUP
# # ============================================================

# import gc
# import ctypes

# try:
#     _libc = ctypes.CDLL("libc.so.6")
#     _malloc_trim = _libc.malloc_trim
#     _malloc_trim.argtypes = [ctypes.c_size_t]
#     _malloc_trim.restype = ctypes.c_int
#     MALLOC_TRIM_AVAILABLE = True
# except Exception:
#     _malloc_trim = None
#     MALLOC_TRIM_AVAILABLE = False


# def cleanup_after_execution():
#     """
#     Release Python objects and return unused heap memory
#     to the operating system.
#     """
    
#     gc.collect()

#     if MALLOC_TRIM_AVAILABLE:
#         _malloc_trim(0)


# # ============================================================
# # 1. TRAINING SETTINGS
# # ============================================================

# TRAINING_SEEDS = [
#     GLOBAL_SEED + i
#     for i in range(N_TRAINING_SEEDS)
# ]

# # IMPORTANT:
# # Previous diagnostics showed severe BLAS/OpenMP
# # oversubscription. Keep this at 1.
# THREAD_LIMIT = 1


# # ============================================================
# # 2. RESULT DIRECTORIES
# # ============================================================

# CV_RESULT_DIR = (
#     RESULTS_DIR / "cv_results"
# )

# CV_RESULT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# CV_ERROR_DIR = (
#     CV_RESULT_DIR / "errors"
# )

# CV_ERROR_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )


# # ============================================================
# # 3. REBUILD CONFIG_ID -> NATIVE CONFIGURATION
# #
# # We DO NOT sample anything here.
# #
# # The configurations already generated in the previous cells
# # are reused exactly.
# # ============================================================

# CONFIG_OBJECTS = {}

# config_counter = 0

# for cell_key in sorted(
#     VALID_CELL_SET,
#     key=str
# ):

#     configs = cell_configurations[
#         cell_key
#     ]

#     for config in configs:

#         config_id = (
#             f"config_{config_counter:05d}"
#         )

#         CONFIG_OBJECTS[
#             config_id
#         ] = config

#         config_counter += 1


# print("=" * 80)
# print("CONFIGURATION OBJECT VERIFICATION")
# print("=" * 80)

# print(
#     f"Configuration objects : "
#     f"{len(CONFIG_OBJECTS)}"
# )

# print(
#     f"Configuration dataframe: "
#     f"{len(configurations_df)}"
# )

# assert (
#     len(CONFIG_OBJECTS)
#     == len(configurations_df)
# ), (
#     "Number of CONFIG_OBJECTS does not match "
#     "configurations_df."
# )

# assert (
#     set(CONFIG_OBJECTS.keys())
#     == set(
#         configurations_df[
#             "config_id"
#         ]
#     )
# ), (
#     "CONFIG_OBJECTS IDs do not match "
#     "configurations_df IDs."
# )

# print(
#     "✓ Existing native ConfigSpace configurations recovered."
# )


# # ============================================================
# # 4. DATASET FEATURE SCHEMAS
# #
# # The original DATASETS contain X with their original feature
# # names.
# #
# # Auto-sklearn's FeatTypeSplit requires the feat_type mapping
# # to use the actual column identifiers supplied to it.
# #
# # Therefore we create a COPY of X with:
# #
# #       0, 1, 2, ..., n-1
# #
# # and create feat_type using exactly those same keys.
# #
# # Original DATASETS are NEVER modified.
# # ============================================================

# CATEGORICAL_INDICES = {

#     # --------------------------------------------------------
#     # CREDIT-G
#     #
#     # 20 features
#     # 13 categorical
#     # 7 numerical
#     # --------------------------------------------------------

#     31: {
#         0,      # checking_status
#         2,      # credit_history
#         3,      # purpose
#         5,      # savings_status
#         6,      # employment
#         8,      # personal_status
#         9,      # other_parties
#         11,     # property_magnitude
#         13,     # other_payment_plans
#         14,     # housing
#         16,     # job
#         18,     # own_telephone
#         19,     # foreign_worker
#     },


#     # --------------------------------------------------------
#     # DIABETES
#     #
#     # 8 numerical
#     # --------------------------------------------------------

#     37: set(),


#     # --------------------------------------------------------
#     # SPAMBASE
#     #
#     # 57 numerical
#     # --------------------------------------------------------

#     44: set(),


#     # --------------------------------------------------------
#     # PHISHING WEBSITES
#     #
#     # 30 categorical
#     # --------------------------------------------------------

#     4534: set(
#         range(30)
#     ),


#     # --------------------------------------------------------
#     # BANK MARKETING
#     #
#     # 16 features
#     # 9 categorical
#     # 7 numerical
#     # --------------------------------------------------------

#     1461: {
#         1,      # job
#         2,      # marital
#         3,      # education
#         4,      # default
#         6,      # housing
#         7,      # loan
#         8,      # contact
#         10,     # month
#         15,     # poutcome
#     },
# }


# EXPECTED_FEATURE_COUNTS = {

#     31: 20,
#     37: 8,
#     44: 57,
#     4534: 30,
#     1461: 16,

# }


# # ============================================================
# # 5. PREPARE DATA FOR AUTO-SKLEARN
# # ============================================================
# #
# # This is the critical fix.
# #
# # Example:
# #
# # Original:
# #
# #     preg | plas | pres | skin | ...
# #
# # becomes:
# #
# #     0 | 1 | 2 | 3 | ...
# #
# # AND:
# #
# #     feat_type = {
# #         0: "numerical",
# #         1: "numerical",
# #         ...
# #     }
# #
# # Thus X columns and feat_type keys ALWAYS MATCH.
# # ============================================================

# DATASET_AUTO_DATA = {}


# def prepare_autosklearn_data(
#     dataset_id
# ):

#     # --------------------------------------------------------
#     # Cache
#     # --------------------------------------------------------

#     if dataset_id in DATASET_AUTO_DATA:

#         cached = (
#             DATASET_AUTO_DATA[
#                 dataset_id
#             ]
#         )

#         return (
#             cached["X"],
#             cached["y"],
#             cached["feat_type"],
#         )


#     # --------------------------------------------------------
#     # Load original data
#     # --------------------------------------------------------

#     data = DATASETS[
#         dataset_id
#     ]

#     X_original = data[
#         "X"
#     ]

#     y = data[
#         "y"
#     ]


#     # --------------------------------------------------------
#     # Feature count
#     # --------------------------------------------------------

#     n_features = (
#         X_original.shape[1]
#     )

#     expected_features = (
#         EXPECTED_FEATURE_COUNTS[
#             dataset_id
#         ]
#     )

#     if n_features != expected_features:

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"expected {expected_features} "
#             f"features but X contains "
#             f"{n_features}."
#         )


#     # --------------------------------------------------------
#     # Categorical indices
#     # --------------------------------------------------------

#     categorical_indices = (
#         CATEGORICAL_INDICES[
#             dataset_id
#         ]
#     )


#     # --------------------------------------------------------
#     # Validate categorical indices
#     # --------------------------------------------------------

#     invalid_indices = (
#         categorical_indices
#         -
#         set(range(n_features))
#     )

#     if invalid_indices:

#         raise ValueError(
#             f"Dataset {dataset_id}: invalid "
#             f"categorical feature indices: "
#             f"{invalid_indices}"
#         )


#     # --------------------------------------------------------
#     # COPY X
#     #
#     # Never mutate DATASETS[id]["X"].
#     # --------------------------------------------------------

#     X = X_original.copy()


#     # --------------------------------------------------------
#     # IMPORTANT:
#     #
#     # Reset the column identifiers to integer positions.
#     # --------------------------------------------------------

#     X.columns = list(
#         range(n_features)
#     )


#     # --------------------------------------------------------
#     # BUILD FEATURE TYPE
#     # --------------------------------------------------------

#     feat_type = {}

#     for column_index in range(
#         n_features
#     ):

#         if (
#             column_index
#             in categorical_indices
#         ):

#             feat_type[
#                 column_index
#             ] = "categorical"

#         else:

#             feat_type[
#                 column_index
#             ] = "numerical"


#     # --------------------------------------------------------
#     # HARD VERIFICATION
#     # --------------------------------------------------------

#     expected_columns = set(
#         range(n_features)
#     )

#     actual_columns = set(
#         X.columns
#     )

#     actual_feat_keys = set(
#         feat_type.keys()
#     )

#     assert (
#         actual_columns
#         == expected_columns
#     ), (
#         f"Dataset {dataset_id}: "
#         f"X columns mismatch. "
#         f"Got {actual_columns}, "
#         f"expected {expected_columns}."
#     )

#     assert (
#         actual_feat_keys
#         == expected_columns
#     ), (
#         f"Dataset {dataset_id}: "
#         f"feat_type keys mismatch. "
#         f"Got {actual_feat_keys}, "
#         f"expected {expected_columns}."
#     )


#     # --------------------------------------------------------
#     # Verify feature-type values
#     # --------------------------------------------------------

#     assert all(
#         value in {
#             "numerical",
#             "categorical",
#         }
#         for value in feat_type.values()
#     )


#     # --------------------------------------------------------
#     # Verify X and y length
#     # --------------------------------------------------------

#     if len(X) != len(y):

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"X has {len(X)} rows but "
#             f"y has {len(y)} rows."
#         )


#     # --------------------------------------------------------
#     # Cache
#     # --------------------------------------------------------

#     DATASET_AUTO_DATA[
#         dataset_id
#     ] = {

#         "X": X,

#         "y": y,

#         "feat_type": feat_type,

#     }


#     return (
#         X,
#         y,
#         feat_type,
#     )


# # ============================================================
# # 6. DATASET PROPERTIES
# # ============================================================

# DATASET_FEAT_TYPES = {}


# def make_dataset_properties(
#     dataset_id
# ):

#     if (
#         dataset_id
#         not in DATASET_FEAT_TYPES
#     ):

#         (
#             _,
#             _,
#             feat_type,
#         ) = prepare_autosklearn_data(
#             dataset_id
#         )

#         DATASET_FEAT_TYPES[
#             dataset_id
#         ] = feat_type


#     feat_type = (
#         DATASET_FEAT_TYPES[
#             dataset_id
#         ]
#     )


#     return {

#         "multilabel": False,

#         "multiclass": False,

#         "binary": True,

#         "sparse": False,

#         "signed": False,

#         "input": feat_type,

#         "output": [
#             "binary"
#         ],

#     }


# # ============================================================
# # 7. VERIFY ALL DATASETS BEFORE ANY TRAINING
# # ============================================================

# print()
# print("=" * 80)
# print("DATASET / FEATURE-TYPE VERIFICATION")
# print("=" * 80)

# for dataset_id in DATASET_IDS:

#     (
#         X_auto,
#         y_auto,
#         feat_type,
#     ) = prepare_autosklearn_data(
#         dataset_id
#     )

#     n_categorical = sum(
#         value == "categorical"
#         for value
#         in feat_type.values()
#     )

#     n_numerical = sum(
#         value == "numerical"
#         for value
#         in feat_type.values()
#     )

#     print()
#     print(
#         f"Dataset {dataset_id}"
#     )

#     print(
#         f"  Shape        : "
#         f"{X_auto.shape}"
#     )

#     print(
#         f"  Columns      : "
#         f"{list(X_auto.columns)}"
#     )

#     print(
#         f"  Numerical    : "
#         f"{n_numerical}"
#     )

#     print(
#         f"  Categorical  : "
#         f"{n_categorical}"
#     )

#     print(
#         f"  feat_type    : "
#         f"{feat_type}"
#     )


# print()
# print(
#     "✓ ALL DATASET FEATURE TYPES VERIFIED."
# )


# # ============================================================
# # 8. DATASET-SPECIFIC PIPELINE CONFIGSPACES
# # ============================================================

# PIPELINE_CONFIG_SPACES = {}


# def get_pipeline_config_space(
#     dataset_id,
#     dataset_properties,
# ):

#     if (
#         dataset_id
#         not in PIPELINE_CONFIG_SPACES
#     ):

#         feat_type = (
#             dataset_properties[
#                 "input"
#             ]
#         )


#         # ----------------------------------------------------
#         # IMPORTANT:
#         #
#         # Create an INSTANCE first.
#         # ----------------------------------------------------

#         pipeline_template = (
#             SimpleClassificationPipeline(
#                 config=None,

#                 feat_type=feat_type,

#                 dataset_properties=
#                     dataset_properties,

#                 include=None,

#                 exclude=None,

#                 random_state=
#                     GLOBAL_SEED,
#             )
#         )


#         # ----------------------------------------------------
#         # INSTANCE METHOD
#         # ----------------------------------------------------

#         config_space = (
#             pipeline_template
#             .get_hyperparameter_search_space(
#                 feat_type=feat_type,

#                 dataset_properties=
#                     dataset_properties,
#             )
#         )


#         PIPELINE_CONFIG_SPACES[
#             dataset_id
#         ] = config_space


#     return PIPELINE_CONFIG_SPACES[
#         dataset_id
#     ]


# # ============================================================
# # 9. CONFIGURATION NORMALIZATION
# # ============================================================
# #
# # Existing configurations can contain:
# #
# #     4.0
# #
# # while target ConfigSpace expects:
# #
# #     4
# #
# # We normalize only the Python representation.
# #
# # We DO NOT resample.
# # ============================================================

# def normalize_configuration_types(
#     config,
#     dataset_id,
#     dataset_properties,
# ):

#     target_cs = (
#         get_pipeline_config_space(
#             dataset_id=
#                 dataset_id,

#             dataset_properties=
#                 dataset_properties,
#         )
#     )


#     original_values = (
#         config.get_dictionary()
#     )


#     normalized_values = {}

#     dropped_inactive = []


#     # --------------------------------------------------------
#     # Iterate over native configuration
#     # --------------------------------------------------------

#     for (
#         name,
#         value
#     ) in original_values.items():


#         # ----------------------------------------------------
#         # Hyperparameter absent from target space
#         # ----------------------------------------------------

#         if name not in target_cs:

#             dropped_inactive.append(
#                 name
#             )

#             continue


#         hp = target_cs[
#             name
#         ]

#         hp_class = (
#             hp.__class__.__name__
#         )


#         # ----------------------------------------------------
#         # INTEGER HYPERPARAMETER
#         # ----------------------------------------------------

#         if hp_class.endswith(
#             "IntegerHyperparameter"
#         ):

#             if isinstance(
#                 value,
#                 (
#                     int,
#                     np.integer,
#                 )
#             ):

#                 normalized_values[
#                     name
#                 ] = int(
#                     value
#                 )


#             elif isinstance(
#                 value,
#                 (
#                     float,
#                     np.floating,
#                 )
#             ):

#                 if not np.isfinite(
#                     value
#                 ):

#                     raise ValueError(
#                         f"Non-finite value "
#                         f"for integer "
#                         f"hyperparameter "
#                         f"'{name}': "
#                         f"{value}"
#                     )


#                 if float(
#                     value
#                 ).is_integer():

#                     normalized_values[
#                         name
#                     ] = int(
#                         value
#                     )

#                 else:

#                     raise ValueError(
#                         f"Cannot convert "
#                         f"{value!r} to integer "
#                         f"hyperparameter "
#                         f"'{name}'."
#                     )


#             else:

#                 raise TypeError(
#                     f"Unexpected type "
#                     f"{type(value)} for "
#                     f"integer "
#                     f"hyperparameter "
#                     f"'{name}'."
#                 )


#         # ----------------------------------------------------
#         # FLOAT HYPERPARAMETER
#         # ----------------------------------------------------

#         elif hp_class.endswith(
#             "FloatHyperparameter"
#         ):

#             normalized_values[
#                 name
#             ] = float(
#                 value
#             )


#         # ----------------------------------------------------
#         # EVERYTHING ELSE
#         # ----------------------------------------------------

#         else:

#             normalized_values[
#                 name
#             ] = value


#     # ========================================================
#     # REBUILD CONFIGURATION
#     # ========================================================

#     normalized_config = (
#         Configuration(
#             configuration_space=
#                 target_cs,

#             values=
#                 normalized_values,
#         )
#     )


#     return (
#         normalized_config,
#         dropped_inactive,
#     )


# # ============================================================
# # 10. BUILD PIPELINE
# # ============================================================

# def build_pipeline(
#     config,
#     dataset_id,
#     dataset_properties,
#     seed,
# ):

#     (
#         normalized_config,
#         dropped_inactive,
#     ) = normalize_configuration_types(
#         config=config,

#         dataset_id=
#             dataset_id,

#         dataset_properties=
#             dataset_properties,
#     )


#     # --------------------------------------------------------
#     # IMPORTANT:
#     #
#     # The exact same feat_type used to construct the
#     # ConfigSpace is passed to the actual pipeline.
#     # --------------------------------------------------------

#     pipeline = (
#         SimpleClassificationPipeline(

#             config=
#                 normalized_config,

#             feat_type=
#                 dataset_properties[
#                     "input"
#                 ],

#             dataset_properties=
#                 dataset_properties,

#             include=None,

#             exclude=None,

#             random_state=
#                 seed,
#         )
#     )


#     return pipeline


# # ============================================================
# # 11. SUBSET HELPER
# #
# # Supports pandas Series and numpy arrays safely.
# # ============================================================

# def subset_vector(
#     vector,
#     indices,
# ):

#     if hasattr(
#         vector,
#         "iloc"
#     ):

#         return vector.iloc[
#             indices
#         ]

#     return np.asarray(
#         vector
#     )[indices]


# # ============================================================
# # 12. SINGLE-FOLD EVALUATION
# # ============================================================

# def evaluate_fold(
#     config,
#     dataset_id,
#     dataset_properties,
#     X,
#     y,
#     train_idx,
#     test_idx,
#     seed,
# ):

#     # --------------------------------------------------------
#     # TRAIN / TEST
#     # --------------------------------------------------------

#     X_train = X.iloc[
#         train_idx
#     ]

#     X_test = X.iloc[
#         test_idx
#     ]


#     y_train = subset_vector(
#         y,
#         train_idx
#     )

#     y_test = subset_vector(
#         y,
#         test_idx
#     )


#     # --------------------------------------------------------
#     # EXPECTED COLUMN POSITIONS
#     # --------------------------------------------------------

#     expected_columns = set(
#         range(
#             X.shape[1]
#         )
#     )


#     # --------------------------------------------------------
#     # X COLUMN CHECK
#     # --------------------------------------------------------

#     if (
#         set(X_train.columns)
#         != expected_columns
#     ):

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"X_train columns are "
#             f"{set(X_train.columns)}, "
#             f"expected "
#             f"{expected_columns}."
#         )


#     if (
#         set(X_test.columns)
#         != expected_columns
#     ):

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"X_test columns are "
#             f"{set(X_test.columns)}, "
#             f"expected "
#             f"{expected_columns}."
#         )


#     # --------------------------------------------------------
#     # FEATURE TYPE CHECK
#     # --------------------------------------------------------

#     feat_type = (
#         dataset_properties[
#             "input"
#         ]
#     )


#     if (
#         set(feat_type.keys())
#         != expected_columns
#     ):

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"feat_type keys are "
#             f"{set(feat_type.keys())}, "
#             f"expected "
#             f"{expected_columns}."
#         )


#     # ========================================================
#     # THREAD-LIMITED PIPELINE EXECUTION
#     # ========================================================

#     with threadpool_limits(
#         limits=THREAD_LIMIT
#     ):

#         # ----------------------------------------------------
#         # BUILD
#         # ----------------------------------------------------

#         pipeline = build_pipeline(
#             config=config,

#             dataset_id=
#                 dataset_id,

#             dataset_properties=
#                 dataset_properties,

#             seed=seed,
#         )


#         # ----------------------------------------------------
#         # FIT
#         # ----------------------------------------------------

#         pipeline.fit(
#             X_train,
#             y_train,
#         )


#         # ----------------------------------------------------
#         # PREDICT PROBABILITIES
#         # ----------------------------------------------------

#         probabilities = (
#             pipeline.predict_proba(
#                 X_test
#             )
#         )


#         probabilities = np.asarray(
#             probabilities
#         )


#         # ----------------------------------------------------
#         # BINARY PROBABILITY CHECK
#         # ----------------------------------------------------

#         if probabilities.ndim != 2:

#             raise ValueError(
#                 f"Dataset {dataset_id}: "
#                 f"predict_proba returned "
#                 f"shape "
#                 f"{probabilities.shape}."
#             )


#         if (
#             probabilities.shape[1]
#             != 2
#         ):

#             raise ValueError(
#                 f"Dataset {dataset_id}: "
#                 f"expected two probability "
#                 f"columns, got "
#                 f"{probabilities.shape}."
#             )


#         positive_probability = (
#             probabilities[:, 1]
#         )


#         # ----------------------------------------------------
#         # NUMERICAL SAFETY
#         # ----------------------------------------------------

#         positive_probability = (
#             np.asarray(
#                 positive_probability,
#                 dtype=float,
#             )
#         )


#         positive_probability = (
#             np.clip(
#                 positive_probability,
#                 1e-15,
#                 1 - 1e-15,
#             )
#         )


#     # ========================================================
#     # LOG LOSS
#     # ========================================================

#     fold_log_loss = (
#         log_loss(
#             y_test,
#             positive_probability,
#             labels=[
#                 0,
#                 1,
#             ],
#         )
#     )


#     return float(
#         fold_log_loss
#     )


# # ============================================================
# # 13. CHECKPOINT LOADER
# # ============================================================

# def load_checkpoint(
#     path
# ):

#     if not path.exists():

#         return pd.DataFrame()


#     try:

#         df = pd.read_csv(
#             path
#         )


#         if df.empty:

#             return pd.DataFrame()


#         return df


#     except Exception as e:

#         print(
#             f"WARNING: Could not read "
#             f"checkpoint {path}: "
#             f"{type(e).__name__}: {e}"
#         )

#         return pd.DataFrame()


# # ============================================================
# # 14. EXECUTION KEY
# # ============================================================

# def make_execution_key(
#     config_id,
#     training_seed,
#     fold_id,
# ):

#     return (
#         str(config_id),
#         int(training_seed),
#         int(fold_id),
#     )


# # ============================================================
# # 15. REMOVE ONLY THE OLD BOGUS UNIFORMINTEGER ERRORS
# # ============================================================
# #
# # These errors were produced by the PREVIOUS broken Cell 11.
# #
# # Dataset 37:
# #     preserve everything.
# #
# # Datasets 31,44,4534,1461:
# #     remove ONLY errors containing both:
# #
# #       "Trying to set illegal value"
# #       "UniformInteger"
# #
# # All other errors remain untouched.
# # ============================================================

# BOGUS_DATASETS = [
#     31,
#     44,
#     4534,
#     1461,
# ]


# def remove_bogus_type_errors(
#     dataset_id
# ):

#     error_path = (
#         CV_ERROR_DIR
#         / f"errors_{dataset_id}.csv"
#     )


#     if not error_path.exists():

#         print(
#             f"[{dataset_id}] "
#             f"No error checkpoint."
#         )

#         return


#     errors = load_checkpoint(
#         error_path
#     )


#     if errors.empty:

#         print(
#             f"[{dataset_id}] "
#             f"Error checkpoint empty."
#         )

#         return


#     if "error" not in errors.columns:

#         print(
#             f"[{dataset_id}] "
#             f"No 'error' column found. "
#             f"Checkpoint left untouched."
#         )

#         return


#     error_text = (
#         errors[
#             "error"
#         ]
#         .fillna("")
#         .astype(str)
#     )


#     bogus_mask = (
#         error_text.str.contains(
#             "Trying to set illegal value",
#             regex=False,
#         )
#         &
#         error_text.str.contains(
#             "UniformInteger",
#             regex=False,
#         )
#     )


#     bogus_count = int(
#         bogus_mask.sum()
#     )


#     if bogus_count == 0:

#         print(
#             f"[{dataset_id}] "
#             f"No old bogus UniformInteger "
#             f"errors found."
#         )

#         return


#     cleaned_errors = (
#         errors.loc[
#             ~bogus_mask
#         ]
#         .copy()
#     )


#     if cleaned_errors.empty:

#         error_path.unlink()

#         print(
#             f"[{dataset_id}] "
#             f"Removed {bogus_count} bogus "
#             f"errors. Checkpoint deleted."
#         )


#     else:

#         cleaned_errors.to_csv(
#             error_path,
#             index=False,
#         )

#         print(
#             f"[{dataset_id}] "
#             f"Removed {bogus_count} bogus "
#             f"errors. Preserved "
#             f"{len(cleaned_errors)} genuine errors."
#         )


# # ============================================================
# # 16. CLEAN OLD CHECKPOINTS
# # ============================================================

# print()
# print("=" * 80)
# print("CLEANING OLD INVALID CONFIGSPACE CHECKPOINTS")
# print("=" * 80)

# for dataset_id in (
#     BOGUS_DATASETS
# ):

#     remove_bogus_type_errors(
#         dataset_id
#     )


# # ============================================================
# # 17. PRE-CHECK — DATA REPRESENTATION
# # ============================================================

# print()
# print("=" * 80)
# print("PRE-CHECK — AUTO-SKLEARN DATA REPRESENTATION")
# print("=" * 80)

# for dataset_id in DATASET_IDS:

#     (
#         X_auto,
#         y_auto,
#         feat_type,
#     ) = prepare_autosklearn_data(
#         dataset_id
#     )


#     expected_columns = set(
#         range(
#             X_auto.shape[1]
#         )
#     )


#     assert (
#         set(X_auto.columns)
#         == expected_columns
#     )


#     assert (
#         set(feat_type.keys())
#         == expected_columns
#     )


#     n_cat = sum(
#         v == "categorical"
#         for v in feat_type.values()
#     )


#     n_num = sum(
#         v == "numerical"
#         for v in feat_type.values()
#     )


#     print()
#     print(
#         f"Dataset {dataset_id}"
#     )

#     print(
#         f"  X shape       : "
#         f"{X_auto.shape}"
#     )

#     print(
#         f"  X columns     : "
#         f"{list(X_auto.columns)}"
#     )

#     print(
#         f"  Numerical     : "
#         f"{n_num}"
#     )

#     print(
#         f"  Categorical   : "
#         f"{n_cat}"
#     )

#     print(
#         f"  Feature types : "
#         f"{feat_type}"
#     )


# print()
# print(
#     "✓ AUTO-SKLEARN DATA REPRESENTATION PASSED."
# )


# # ============================================================
# # 18. PRE-CHECK — CONFIGURATION + PIPELINE
# #
# # Dataset 31 / config_00000 is checked before any full
# # experiment execution.
# # ============================================================

# print()
# print("=" * 80)
# print("PRE-CHECK — DATASET 31 / CONFIG_00000")
# print("=" * 80)

# test_dataset_id = 31


# (
#     test_X,
#     test_y,
#     test_feat_type,
# ) = prepare_autosklearn_data(
#     test_dataset_id
# )


# test_dataset_properties = (
#     make_dataset_properties(
#         test_dataset_id
#     )
# )


# test_config = (
#     CONFIG_OBJECTS[
#         "config_00000"
#     ]
# )


# # ------------------------------------------------------------
# # Construct target ConfigSpace
# # ------------------------------------------------------------

# test_target_cs = (
#     get_pipeline_config_space(
#         dataset_id=
#             test_dataset_id,

#         dataset_properties=
#             test_dataset_properties,
#     )
# )


# print(
#     f"Target ConfigSpace hyperparameters: "
#     f"{len(test_target_cs.get_hyperparameters())}"
# )


# # ------------------------------------------------------------
# # Normalize config
# # ------------------------------------------------------------

# (
#     test_normalized_config,
#     test_dropped,
# ) = normalize_configuration_types(
#     config=
#         test_config,

#     dataset_id=
#         test_dataset_id,

#     dataset_properties=
#         test_dataset_properties,
# )


# print(
#     f"Original config parameters : "
#     f"{len(test_config.get_dictionary())}"
# )


# print(
#     f"Normalized config parameters: "
#     f"{len(test_normalized_config.get_dictionary())}"
# )


# print(
#     f"Inactive parameters omitted : "
#     f"{len(test_dropped)}"
# )


# # ------------------------------------------------------------
# # Build pipeline
# # ------------------------------------------------------------

# test_pipeline = build_pipeline(
#     config=
#         test_config,

#     dataset_id=
#         test_dataset_id,

#     dataset_properties=
#         test_dataset_properties,

#     seed=
#         GLOBAL_SEED,
# )


# print()
# print(
#     "✓ Target ConfigSpace constructed."
# )

# print(
#     "✓ Configuration normalized."
# )

# print(
#     "✓ Pipeline constructed."
# )

# print(
#     "✓ Integer-indexed feat_type supplied."
# )


# # ============================================================
# # 19. PRE-CHECK — ACTUAL FIT
# #
# # This is deliberately one fold only.
# #
# # If this succeeds, the exact previous error path has been
# # eliminated before the 24,750-fold dataset loop starts.
# # ============================================================

# print()
# print("=" * 80)
# print("PRE-CHECK — ACTUAL DATASET 31 FIT")
# print("=" * 80)


# test_fold = (
#     CV_SPLITS[
#         test_dataset_id
#     ][0]
# )


# test_train_idx = (
#     test_fold[
#         "train_idx"
#     ]
# )


# test_test_idx = (
#     test_fold[
#         "test_idx"
#     ]
# )


# print(
#     f"Train samples : "
#     f"{len(test_train_idx)}"
# )


# print(
#     f"Test samples  : "
#     f"{len(test_test_idx)}"
# )


# print(
#     "Running one actual fit/predict..."
# )


# test_log_loss = (
#     evaluate_fold(
#         config=
#             test_config,

#         dataset_id=
#             test_dataset_id,

#         dataset_properties=
#             test_dataset_properties,

#         X=
#             test_X,

#         y=
#             test_y,

#         train_idx=
#             test_train_idx,

#         test_idx=
#             test_test_idx,

#         seed=
#             GLOBAL_SEED,
#     )
# )


# print()
# print(
#     f"✓ ACTUAL PRE-CHECK PASSED."
# )

# print(
#     f"Test fold log-loss: "
#     f"{test_log_loss:.8f}"
# )


# # ============================================================
# # 20. FULL EXPERIMENT
# # ============================================================

# ALL_DATASET_RESULTS = {}


# for dataset_id in DATASET_IDS:

#     dataset_name = (
#         DATASETS[
#             dataset_id
#         ][
#             "name"
#         ]
#     )


#     print("\n\n")
#     print("=" * 80)

#     print(
#         f"DATASET {dataset_id} — "
#         f"{dataset_name}"
#     )

#     print("=" * 80)


#     # --------------------------------------------------------
#     # DATA
#     #
#     # IMPORTANT:
#     # Use PREPARED X, not DATASETS[id]["X"] directly.
#     # --------------------------------------------------------

#     (
#         X,
#         y,
#         feat_type,
#     ) = prepare_autosklearn_data(
#         dataset_id
#     )


#     # --------------------------------------------------------
#     # DATASET PROPERTIES
#     # --------------------------------------------------------

#     dataset_properties = (
#         make_dataset_properties(
#             dataset_id
#         )
#     )


#     # --------------------------------------------------------
#     # CHECKPOINT PATHS
#     # --------------------------------------------------------

#     result_path = (
#         CV_RESULT_DIR
#         / f"cv_results_{dataset_id}.csv"
#     )


#     error_path = (
#         CV_ERROR_DIR
#         / f"errors_{dataset_id}.csv"
#     )


#     # --------------------------------------------------------
#     # LOAD CHECKPOINTS
#     # --------------------------------------------------------

#     existing_results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     existing_errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     # --------------------------------------------------------
#     # DATASET-SPECIFIC CONFIGSPACE
#     # --------------------------------------------------------

#     target_config_space = (
#         get_pipeline_config_space(
#             dataset_id=
#                 dataset_id,

#             dataset_properties=
#                 dataset_properties,
#         )
#     )


#     print(
#         f"Target ConfigSpace ready."
#     )


#     print(
#         f"Hyperparameters: "
#         f"{len(target_config_space.get_hyperparameters())}"
#     )


#     # --------------------------------------------------------
#     # FIXED CV SPLITS
#     # --------------------------------------------------------

#     folds = CV_SPLITS[
#         dataset_id
#     ]


#     if len(folds) != N_CV_FOLDS:

#         raise ValueError(
#             f"Dataset {dataset_id}: "
#             f"expected {N_CV_FOLDS} folds, "
#             f"got {len(folds)}."
#         )


#     # --------------------------------------------------------
#     # EXPECTED EXECUTIONS
#     # --------------------------------------------------------

#     total_folds = (
#         len(configurations_df)
#         *
#         len(TRAINING_SEEDS)
#         *
#         N_CV_FOLDS
#     )


#     # --------------------------------------------------------
#     # COMPLETED EXECUTION KEYS
#     # --------------------------------------------------------

#     completed_keys = set()


#     # --------------------------------------------------------
#     # SUCCESSFUL EXECUTIONS
#     # --------------------------------------------------------

#     if not existing_results.empty:

#         required_result_columns = {
#             "config_id",
#             "training_seed",
#             "fold",
#         }

#         if required_result_columns.issubset(
#             existing_results.columns
#         ):

#             for _, row in (
#                 existing_results.iterrows()
#             ):

#                 completed_keys.add(
#                     make_execution_key(
#                         row[
#                             "config_id"
#                         ],

#                         row[
#                             "training_seed"
#                         ],

#                         row[
#                             "fold"
#                         ],
#                     )
#                 )


#     # --------------------------------------------------------
#     # GENUINE FAILURES
#     # --------------------------------------------------------

#     if not existing_errors.empty:

#         required_error_columns = {
#             "config_id",
#             "training_seed",
#             "fold",
#         }

#         if required_error_columns.issubset(
#             existing_errors.columns
#         ):

#             for _, row in (
#                 existing_errors.iterrows()
#             ):

#                 completed_keys.add(
#                     make_execution_key(
#                         row[
#                             "config_id"
#                         ],

#                         row[
#                             "training_seed"
#                         ],

#                         row[
#                             "fold"
#                         ],
#                     )
#                 )


#     # --------------------------------------------------------
#     # DATASET START
#     # --------------------------------------------------------

#     dataset_start = time.time()


#     print()
#     print(
#         f"Existing successful folds : "
#         f"{len(existing_results)}"
#     )

#     print(
#         f"Existing genuine failures : "
#         f"{len(existing_errors)}"
#     )

#     print(
#         f"Unique completed          : "
#         f"{len(completed_keys)}"
#     )

#     print(
#         f"Expected executions       : "
#         f"{total_folds}"
#     )

#     print(
#         f"Remaining executions      : "
#         f"{total_folds - len(completed_keys)}"
#     )


#     # ========================================================
#     # CONFIGURATION LOOP
#     # ========================================================

#     for config_number, (
#         _,
#         config_row
#     ) in enumerate(
#         configurations_df.iterrows(),
#         start=1,
#     ):

#         config_id = (
#             config_row[
#                 "config_id"
#             ]
#         )


#         # ----------------------------------------------------
#         # EXISTING NATIVE CONFIGURATION
#         # ----------------------------------------------------

#         config = (
#             CONFIG_OBJECTS[
#                 config_id
#             ]
#         )


#         # ----------------------------------------------------
#         # TRAINING SEEDS
#         # ----------------------------------------------------

#         for training_seed in (
#             TRAINING_SEEDS
#         ):


#             # ------------------------------------------------
#             # FIXED FOLDS
#             # ------------------------------------------------

#             for fold_info in folds:

#                 fold_id = (
#                     fold_info[
#                         "fold"
#                     ]
#                 )


#                 execution_key = (
#                     make_execution_key(
#                         config_id=
#                             config_id,

#                         training_seed=
#                             training_seed,

#                         fold_id=
#                             fold_id,
#                     )
#                 )


#                 # ============================================
#                 # RESUME
#                 # ============================================

#                 if (
#                     execution_key
#                     in completed_keys
#                 ):

#                     continue


    


#                 fold_start = time.time()


#                 # ============================================
#                 # EXECUTION
#                 # ============================================

#                 try:

#                     fold_log_loss = (
#                         evaluate_fold(

#                             config=
#                                 config,

#                             dataset_id=
#                                 dataset_id,

#                             dataset_properties=
#                                 dataset_properties,

#                             X=
#                                 X,

#                             y=
#                                 y,

#                             train_idx=
#                                 fold_info[
#                                     "train_idx"
#                                 ],

#                             test_idx=
#                                 fold_info[
#                                     "test_idx"
#                                 ],

#                             seed=
#                                 training_seed,
#                         )
#                     )


#                     # ========================================
#                     # SUCCESS ROW
#                     # ========================================

#                     result_row = {

#                         "dataset_id":
#                             dataset_id,

#                         "dataset":
#                             dataset_name,

#                         "config_id":
#                             config_id,

#                         "classifier":
#                             config_row[
#                                 "classifier"
#                             ],

#                         "feature_preprocessor":
#                             config_row[
#                                 "feature_preprocessor"
#                             ],

#                         "balancing":
#                             config_row[
#                                 "balancing"
#                             ],

#                         "config_replicate":
#                             config_row[
#                                 "config_replicate"
#                             ],

#                         "training_seed":
#                             training_seed,

#                         "fold":
#                             fold_id,

#                         "log_loss":
#                             fold_log_loss,

#                         "status":
#                             "success",

#                         "runtime_seconds":
#                             (
#                                 time.time()
#                                 -
#                                 fold_start
#                             ),
#                     }


#                     # ========================================
#                     # IMMEDIATE CHECKPOINT
#                     # ========================================

#                     pd.DataFrame(
#                         [result_row]
#                     ).to_csv(

#                         result_path,

#                         mode="a",

#                         header=(
#                             not result_path.exists()
#                         ),

#                         index=False,
#                     )


#                     completed_keys.add(
#                         execution_key
#                     )


#                 # ============================================
#                 # FAILURE
#                 # ============================================

#                 except Exception as e:

#                     error_row = {

#                         "dataset_id":
#                             dataset_id,

#                         "dataset":
#                             dataset_name,

#                         "config_id":
#                             config_id,

#                         "classifier":
#                             config_row[
#                                 "classifier"
#                             ],

#                         "feature_preprocessor":
#                             config_row[
#                                 "feature_preprocessor"
#                             ],

#                         "balancing":
#                             config_row[
#                                 "balancing"
#                             ],

#                         "config_replicate":
#                             config_row[
#                                 "config_replicate"
#                             ],

#                         "training_seed":
#                             training_seed,

#                         "fold":
#                             fold_id,

#                         "error_type":
#                             type(e).__name__,

#                         "error":
#                             repr(e),

#                         "runtime_seconds":
#                             (
#                                 time.time()
#                                 -
#                                 fold_start
#                             ),

#                         "status":
#                             "failed",
#                     }


#                     # ========================================
#                     # IMMEDIATE ERROR CHECKPOINT
#                     # ========================================

#                     pd.DataFrame(
#                         [error_row]
#                     ).to_csv(

#                         error_path,

#                         mode="a",

#                         header=(
#                             not error_path.exists()
#                         ),

#                         index=False,
#                     )


#                     completed_keys.add(
#                         execution_key
#                     )


#                     # ========================================
#                     # PRINT FAILURE
#                     # ========================================

#                     print(
#                         f"[{dataset_id}] "
#                         f"FAIL | "
#                         f"{config_id} | "
#                         f"{config_row['classifier']} | "
#                         f"{config_row['feature_preprocessor']} | "
#                         f"{config_row['balancing']} | "
#                         f"seed={training_seed} | "
#                         f"fold={fold_id} | "
#                         f"{type(e).__name__}: "
#                         f"{e}"
#                     )


#                 # ============================================
#                 # PROGRESS
#                 # ============================================

#                 completed_count = len(
#                     completed_keys
#                 )


#                 if (
#                     completed_count % 25 == 0
#                     or
#                     completed_count
#                     == total_folds
#                 ):

#                     elapsed = (
#                         time.time()
#                         -
#                         dataset_start
#                     )


#                     remaining = (
#                         total_folds
#                         -
#                         completed_count
#                     )


#                     rate = (
#                         completed_count
#                         /
#                         max(
#                             elapsed / 60,
#                             1e-9,
#                         )
#                     )


#                     eta_minutes = (
#                         remaining / rate
#                         if rate > 0
#                         else np.nan
#                     )


#                     print(
#                         f"[{dataset_id}] "
#                         f"{completed_count}/"
#                         f"{total_folds} "
#                         f"("
#                         f"{100 * completed_count / total_folds:.2f}%"
#                         f") | "
#                         f"config "
#                         f"{config_number}/"
#                         f"{len(configurations_df)} | "
#                         f"elapsed "
#                         f"{elapsed / 60:.1f} min | "
#                         f"ETA "
#                         f"{eta_minutes / 60:.1f} h"
#                     )


#     # ========================================================
#     # DATASET COMPLETE
#     # ========================================================

#     final_results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     final_errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     ALL_DATASET_RESULTS[
#         dataset_id
#     ] = final_results


#     # ========================================================
#     # DATASET SUMMARY
#     # ========================================================

#     successful_count = (
#         len(final_results)
#     )


#     failed_count = (
#         len(final_errors)
#     )


#     attempted_count = (
#         successful_count
#         +
#         failed_count
#     )


#     print()
#     print("-" * 80)

#     print(
#         f"DATASET {dataset_id} COMPLETE"
#     )

#     print("-" * 80)

#     print(
#         f"Successful folds : "
#         f"{successful_count}"
#     )

#     print(
#         f"Failed folds     : "
#         f"{failed_count}"
#     )

#     print(
#         f"Total attempted  : "
#         f"{attempted_count}"
#     )

#     print(
#         f"Expected folds   : "
#         f"{total_folds}"
#     )


#     # ========================================================
#     # STRICT COMPLETENESS
#     # ========================================================

#     if (
#         attempted_count
#         != total_folds
#     ):

#         missing = (
#             total_folds
#             -
#             attempted_count
#         )

#         raise RuntimeError(
#             f"DATASET {dataset_id} "
#             f"IS NOT COMPLETE. "
#             f"Missing {missing} executions."
#         )


#     print(
#         f"✓ Dataset {dataset_id} "
#         f"has exactly "
#         f"{total_folds} "
#         f"attempted executions."
#     )


# # ============================================================
# # 21. GLOBAL SUMMARY
# # ============================================================

# print("\n\n")
# print("=" * 80)
# print("FULL CV EXPERIMENT COMPLETE")
# print("=" * 80)


# grand_success = 0
# grand_failure = 0


# for dataset_id in DATASET_IDS:

#     dataset_name = (
#         DATASETS[
#             dataset_id
#         ][
#             "name"
#         ]
#     )


#     result_path = (
#         CV_RESULT_DIR
#         /
#         f"cv_results_{dataset_id}.csv"
#     )


#     error_path = (
#         CV_ERROR_DIR
#         /
#         f"errors_{dataset_id}.csv"
#     )


#     results = (
#         load_checkpoint(
#             result_path
#         )
#     )


#     errors = (
#         load_checkpoint(
#             error_path
#         )
#     )


#     success_count = len(
#         results
#     )


#     failure_count = len(
#         errors
#     )


#     attempted = (
#         success_count
#         +
#         failure_count
#     )


#     grand_success += (
#         success_count
#     )


#     grand_failure += (
#         failure_count
#     )


#     print(
#         f"{dataset_id} | "
#         f"{dataset_name:20s} | "
#         f"{success_count:6d} successful | "
#         f"{failure_count:6d} failed | "
#         f"{attempted:6d} attempted"
#     )


# # ============================================================
# # GLOBAL TOTALS
# # ============================================================

# grand_total = (
#     grand_success
#     +
#     grand_failure
# )


# grand_expected = (
#     len(DATASET_IDS)
#     *
#     len(configurations_df)
#     *
#     len(TRAINING_SEEDS)
#     *
#     N_CV_FOLDS
# )


# print("-" * 80)


# print(
#     f"TOTAL SUCCESSFUL : "
#     f"{grand_success}"
# )


# print(
#     f"TOTAL FAILED     : "
#     f"{grand_failure}"
# )


# print(
#     f"TOTAL ATTEMPTED  : "
#     f"{grand_total}"
# )


# print(
#     f"TOTAL EXPECTED   : "
#     f"{grand_expected}"
# )


# print("-" * 80)


# assert (
#     grand_total
#     ==
#     grand_expected
# ), (
#     f"Global completeness failure: "
#     f"{grand_total} attempted vs "
#     f"{grand_expected} expected."
# )


# print(
#     "✓ ALL DATASETS HAVE THE EXPECTED "
#     "NUMBER OF ATTEMPTED EXECUTIONS."
# )


# print(
#     "✓ CV EXPERIMENT FINISHED."
# )

In [17]:
# ============================================================
# CELL 11 — FULL CONTROLLED CV EXPERIMENT
#
# FINAL FIXED VERSION
#
# PURPOSE
# -------
# Run the complete controlled Auto-sklearn CV experiment using
# the EXISTING native ConfigSpace configurations.
#
# IMPORTANT:
#   1. NO new configurations are sampled here.
#   2. Existing cell_configurations are reused.
#   3. Original X column names are converted to integer
#      positions ONLY for the Auto-sklearn representation.
#   4. feat_type uses exactly the same integer column positions.
#   5. Dataset categorical/numerical structure is explicitly
#      defined for the five datasets.
#   6. The same transformed X representation is used in:
#          - ConfigSpace construction
#          - pipeline construction
#          - fit
#          - prediction
#   7. Integer ConfigSpace values such as 4.0 -> 4 are fixed.
#   8. Threading is limited to 1 to prevent oversubscription.
#   9. Existing successful results are resumed.
#  10. Existing OLD UniformInteger construction failures are
#      removed; genuine failures are preserved.
#  11. No arbitrary timeout/skip is introduced.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import time
import traceback



import numpy as np
import pandas as pd

from sklearn.metrics import log_loss

from threadpoolctl import threadpool_limits

from ConfigSpace import Configuration

from autosklearn.pipeline.classification import (
    SimpleClassificationPipeline
)


# ============================================================
# MEMORY CLEANUP
# ============================================================

import gc
import ctypes

try:
    _libc = ctypes.CDLL("libc.so.6")
    _malloc_trim = _libc.malloc_trim
    _malloc_trim.argtypes = [ctypes.c_size_t]
    _malloc_trim.restype = ctypes.c_int
    MALLOC_TRIM_AVAILABLE = True
except Exception:
    _malloc_trim = None
    MALLOC_TRIM_AVAILABLE = False


# Full malloc_trim is deliberately not called after EVERY fold.
# It can itself add overhead. Python GC runs every fold, while
# unused native heap memory is returned to the OS periodically.
CLEANUP_COUNTER = 0
MALLOC_TRIM_EVERY = 10


def cleanup_after_execution():
    """
    Release Python objects after every fold.

    Every fold:
        - run Python garbage collection.

    Every MALLOC_TRIM_EVERY folds:
        - return unused glibc heap memory to the OS.

    This does NOT change the experiment, configurations, folds,
    seeds, timeout policy, or execution order.
    """

    global CLEANUP_COUNTER

    CLEANUP_COUNTER += 1

    gc.collect()

    if (
        MALLOC_TRIM_AVAILABLE
        and CLEANUP_COUNTER % MALLOC_TRIM_EVERY == 0
    ):
        _malloc_trim(0)


# ============================================================
# 1. TRAINING SETTINGS
# ============================================================

TRAINING_SEEDS = [
    GLOBAL_SEED + i
    for i in range(N_TRAINING_SEEDS)
]

# IMPORTANT:
# Previous diagnostics showed severe BLAS/OpenMP
# oversubscription. Keep this at 1.
THREAD_LIMIT = 1


# ============================================================
# 2. RESULT DIRECTORIES
# ============================================================

CV_RESULT_DIR = (
    RESULTS_DIR / "cv_results"
)

CV_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CV_ERROR_DIR = (
    CV_RESULT_DIR / "errors"
)

CV_ERROR_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. REBUILD CONFIG_ID -> NATIVE CONFIGURATION
#
# We DO NOT sample anything here.
#
# The configurations already generated in the previous cells
# are reused exactly.
# ============================================================

CONFIG_OBJECTS = {}

config_counter = 0

for cell_key in sorted(
    VALID_CELL_SET,
    key=str
):

    configs = cell_configurations[
        cell_key
    ]

    for config in configs:

        config_id = (
            f"config_{config_counter:05d}"
        )

        CONFIG_OBJECTS[
            config_id
        ] = config

        config_counter += 1


print("=" * 80)
print("CONFIGURATION OBJECT VERIFICATION")
print("=" * 80)

print(
    f"Configuration objects : "
    f"{len(CONFIG_OBJECTS)}"
)

print(
    f"Configuration dataframe: "
    f"{len(configurations_df)}"
)

assert (
    len(CONFIG_OBJECTS)
    == len(configurations_df)
), (
    "Number of CONFIG_OBJECTS does not match "
    "configurations_df."
)

assert (
    set(CONFIG_OBJECTS.keys())
    == set(
        configurations_df[
            "config_id"
        ]
    )
), (
    "CONFIG_OBJECTS IDs do not match "
    "configurations_df IDs."
)

print(
    "✓ Existing native ConfigSpace configurations recovered."
)


# ============================================================
# 4. DATASET FEATURE SCHEMAS
#
# The original DATASETS contain X with their original feature
# names.
#
# Auto-sklearn's FeatTypeSplit requires the feat_type mapping
# to use the actual column identifiers supplied to it.
#
# Therefore we create a COPY of X with:
#
#       0, 1, 2, ..., n-1
#
# and create feat_type using exactly those same keys.
#
# Original DATASETS are NEVER modified.
# ============================================================

CATEGORICAL_INDICES = {

    # --------------------------------------------------------
    # CREDIT-G
    #
    # 20 features
    # 13 categorical
    # 7 numerical
    # --------------------------------------------------------

    31: {
        0,      # checking_status
        2,      # credit_history
        3,      # purpose
        5,      # savings_status
        6,      # employment
        8,      # personal_status
        9,      # other_parties
        11,     # property_magnitude
        13,     # other_payment_plans
        14,     # housing
        16,     # job
        18,     # own_telephone
        19,     # foreign_worker
    },


    # --------------------------------------------------------
    # DIABETES
    #
    # 8 numerical
    # --------------------------------------------------------

    37: set(),


    # --------------------------------------------------------
    # SPAMBASE
    #
    # 57 numerical
    # --------------------------------------------------------

    44: set(),


    # --------------------------------------------------------
    # PHISHING WEBSITES
    #
    # 30 categorical
    # --------------------------------------------------------

    4534: set(
        range(30)
    ),


    # --------------------------------------------------------
    # BANK MARKETING
    #
    # 16 features
    # 9 categorical
    # 7 numerical
    # --------------------------------------------------------

    1461: {
        1,      # job
        2,      # marital
        3,      # education
        4,      # default
        6,      # housing
        7,      # loan
        8,      # contact
        10,     # month
        15,     # poutcome
    },
}


EXPECTED_FEATURE_COUNTS = {

    31: 20,
    37: 8,
    44: 57,
    4534: 30,
    1461: 16,

}


# ============================================================
# 5. PREPARE DATA FOR AUTO-SKLEARN
# ============================================================
#
# This is the critical fix.
#
# Example:
#
# Original:
#
#     preg | plas | pres | skin | ...
#
# becomes:
#
#     0 | 1 | 2 | 3 | ...
#
# AND:
#
#     feat_type = {
#         0: "numerical",
#         1: "numerical",
#         ...
#     }
#
# Thus X columns and feat_type keys ALWAYS MATCH.
# ============================================================

DATASET_AUTO_DATA = {}


def prepare_autosklearn_data(
    dataset_id
):

    # --------------------------------------------------------
    # Cache
    # --------------------------------------------------------

    if dataset_id in DATASET_AUTO_DATA:

        cached = (
            DATASET_AUTO_DATA[
                dataset_id
            ]
        )

        return (
            cached["X"],
            cached["y"],
            cached["feat_type"],
        )


    # --------------------------------------------------------
    # Load original data
    # --------------------------------------------------------

    data = DATASETS[
        dataset_id
    ]

    X_original = data[
        "X"
    ]

    y = data[
        "y"
    ]


    # --------------------------------------------------------
    # Feature count
    # --------------------------------------------------------

    n_features = (
        X_original.shape[1]
    )

    expected_features = (
        EXPECTED_FEATURE_COUNTS[
            dataset_id
        ]
    )

    if n_features != expected_features:

        raise ValueError(
            f"Dataset {dataset_id}: "
            f"expected {expected_features} "
            f"features but X contains "
            f"{n_features}."
        )


    # --------------------------------------------------------
    # Categorical indices
    # --------------------------------------------------------

    categorical_indices = (
        CATEGORICAL_INDICES[
            dataset_id
        ]
    )


    # --------------------------------------------------------
    # Validate categorical indices
    # --------------------------------------------------------

    invalid_indices = (
        categorical_indices
        -
        set(range(n_features))
    )

    if invalid_indices:

        raise ValueError(
            f"Dataset {dataset_id}: invalid "
            f"categorical feature indices: "
            f"{invalid_indices}"
        )


    # --------------------------------------------------------
    # COPY X
    #
    # Never mutate DATASETS[id]["X"].
    # --------------------------------------------------------

    X = X_original.copy()


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Reset the column identifiers to integer positions.
    # --------------------------------------------------------

    X.columns = list(
        range(n_features)
    )


    # --------------------------------------------------------
    # BUILD FEATURE TYPE
    # --------------------------------------------------------

    feat_type = {}

    for column_index in range(
        n_features
    ):

        if (
            column_index
            in categorical_indices
        ):

            feat_type[
                column_index
            ] = "categorical"

        else:

            feat_type[
                column_index
            ] = "numerical"


    # --------------------------------------------------------
    # HARD VERIFICATION
    # --------------------------------------------------------

    expected_columns = set(
        range(n_features)
    )

    actual_columns = set(
        X.columns
    )

    actual_feat_keys = set(
        feat_type.keys()
    )

    assert (
        actual_columns
        == expected_columns
    ), (
        f"Dataset {dataset_id}: "
        f"X columns mismatch. "
        f"Got {actual_columns}, "
        f"expected {expected_columns}."
    )

    assert (
        actual_feat_keys
        == expected_columns
    ), (
        f"Dataset {dataset_id}: "
        f"feat_type keys mismatch. "
        f"Got {actual_feat_keys}, "
        f"expected {expected_columns}."
    )


    # --------------------------------------------------------
    # Verify feature-type values
    # --------------------------------------------------------

    assert all(
        value in {
            "numerical",
            "categorical",
        }
        for value in feat_type.values()
    )


    # --------------------------------------------------------
    # Verify X and y length
    # --------------------------------------------------------

    if len(X) != len(y):

        raise ValueError(
            f"Dataset {dataset_id}: "
            f"X has {len(X)} rows but "
            f"y has {len(y)} rows."
        )


    # --------------------------------------------------------
    # Cache
    # --------------------------------------------------------

    DATASET_AUTO_DATA[
        dataset_id
    ] = {

        "X": X,

        "y": y,

        "feat_type": feat_type,

    }


    return (
        X,
        y,
        feat_type,
    )


# ============================================================
# 6. DATASET PROPERTIES
# ============================================================

DATASET_FEAT_TYPES = {}


def make_dataset_properties(
    dataset_id
):

    if (
        dataset_id
        not in DATASET_FEAT_TYPES
    ):

        (
            _,
            _,
            feat_type,
        ) = prepare_autosklearn_data(
            dataset_id
        )

        DATASET_FEAT_TYPES[
            dataset_id
        ] = feat_type


    feat_type = (
        DATASET_FEAT_TYPES[
            dataset_id
        ]
    )


    return {

        "multilabel": False,

        "multiclass": False,

        "binary": True,

        "sparse": False,

        "signed": False,

        "input": feat_type,

        "output": [
            "binary"
        ],

    }


# ============================================================
# 7. VERIFY ALL DATASETS BEFORE ANY TRAINING
# ============================================================

print()
print("=" * 80)
print("DATASET / FEATURE-TYPE VERIFICATION")
print("=" * 80)

for dataset_id in DATASET_IDS:

    (
        X_auto,
        y_auto,
        feat_type,
    ) = prepare_autosklearn_data(
        dataset_id
    )

    n_categorical = sum(
        value == "categorical"
        for value
        in feat_type.values()
    )

    n_numerical = sum(
        value == "numerical"
        for value
        in feat_type.values()
    )

    print()
    print(
        f"Dataset {dataset_id}"
    )

    print(
        f"  Shape        : "
        f"{X_auto.shape}"
    )

    print(
        f"  Columns      : "
        f"{list(X_auto.columns)}"
    )

    print(
        f"  Numerical    : "
        f"{n_numerical}"
    )

    print(
        f"  Categorical  : "
        f"{n_categorical}"
    )

    print(
        f"  feat_type    : "
        f"{feat_type}"
    )


print()
print(
    "✓ ALL DATASET FEATURE TYPES VERIFIED."
)


# ============================================================
# 8. DATASET-SPECIFIC PIPELINE CONFIGSPACES
# ============================================================

PIPELINE_CONFIG_SPACES = {}


def get_pipeline_config_space(
    dataset_id,
    dataset_properties,
):

    if (
        dataset_id
        not in PIPELINE_CONFIG_SPACES
    ):

        feat_type = (
            dataset_properties[
                "input"
            ]
        )


        # ----------------------------------------------------
        # IMPORTANT:
        #
        # Create an INSTANCE first.
        # ----------------------------------------------------

        pipeline_template = (
            SimpleClassificationPipeline(
                config=None,

                feat_type=feat_type,

                dataset_properties=
                    dataset_properties,

                include=None,

                exclude=None,

                random_state=
                    GLOBAL_SEED,
            )
        )


        # ----------------------------------------------------
        # INSTANCE METHOD
        # ----------------------------------------------------

        config_space = (
            pipeline_template
            .get_hyperparameter_search_space(
                feat_type=feat_type,

                dataset_properties=
                    dataset_properties,
            )
        )


        PIPELINE_CONFIG_SPACES[
            dataset_id
        ] = config_space


    return PIPELINE_CONFIG_SPACES[
        dataset_id
    ]


# ============================================================
# 9. CONFIGURATION NORMALIZATION
# ============================================================
#
# Existing configurations can contain:
#
#     4.0
#
# while target ConfigSpace expects:
#
#     4
#
# We normalize only the Python representation.
#
# We DO NOT resample.
# ============================================================

def normalize_configuration_types(
    config,
    dataset_id,
    dataset_properties,
):

    target_cs = (
        get_pipeline_config_space(
            dataset_id=
                dataset_id,

            dataset_properties=
                dataset_properties,
        )
    )


    original_values = (
        config.get_dictionary()
    )


    normalized_values = {}

    dropped_inactive = []


    # --------------------------------------------------------
    # Iterate over native configuration
    # --------------------------------------------------------

    for (
        name,
        value
    ) in original_values.items():


        # ----------------------------------------------------
        # Hyperparameter absent from target space
        # ----------------------------------------------------

        if name not in target_cs:

            dropped_inactive.append(
                name
            )

            continue


        hp = target_cs[
            name
        ]

        hp_class = (
            hp.__class__.__name__
        )


        # ----------------------------------------------------
        # INTEGER HYPERPARAMETER
        # ----------------------------------------------------

        if hp_class.endswith(
            "IntegerHyperparameter"
        ):

            if isinstance(
                value,
                (
                    int,
                    np.integer,
                )
            ):

                normalized_values[
                    name
                ] = int(
                    value
                )


            elif isinstance(
                value,
                (
                    float,
                    np.floating,
                )
            ):

                if not np.isfinite(
                    value
                ):

                    raise ValueError(
                        f"Non-finite value "
                        f"for integer "
                        f"hyperparameter "
                        f"'{name}': "
                        f"{value}"
                    )


                if float(
                    value
                ).is_integer():

                    normalized_values[
                        name
                    ] = int(
                        value
                    )

                else:

                    raise ValueError(
                        f"Cannot convert "
                        f"{value!r} to integer "
                        f"hyperparameter "
                        f"'{name}'."
                    )


            else:

                raise TypeError(
                    f"Unexpected type "
                    f"{type(value)} for "
                    f"integer "
                    f"hyperparameter "
                    f"'{name}'."
                )


        # ----------------------------------------------------
        # FLOAT HYPERPARAMETER
        # ----------------------------------------------------

        elif hp_class.endswith(
            "FloatHyperparameter"
        ):

            normalized_values[
                name
            ] = float(
                value
            )


        # ----------------------------------------------------
        # EVERYTHING ELSE
        # ----------------------------------------------------

        else:

            normalized_values[
                name
            ] = value


    # ========================================================
    # REBUILD CONFIGURATION
    # ========================================================

    normalized_config = (
        Configuration(
            configuration_space=
                target_cs,

            values=
                normalized_values,
        )
    )


    return (
        normalized_config,
        dropped_inactive,
    )


# ============================================================
# 10. BUILD PIPELINE
# ============================================================

def build_pipeline(
    config,
    dataset_id,
    dataset_properties,
    seed,
):

    (
        normalized_config,
        dropped_inactive,
    ) = normalize_configuration_types(
        config=config,

        dataset_id=
            dataset_id,

        dataset_properties=
            dataset_properties,
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # The exact same feat_type used to construct the
    # ConfigSpace is passed to the actual pipeline.
    # --------------------------------------------------------

    pipeline = (
        SimpleClassificationPipeline(

            config=
                normalized_config,

            feat_type=
                dataset_properties[
                    "input"
                ],

            dataset_properties=
                dataset_properties,

            include=None,

            exclude=None,

            random_state=
                seed,
        )
    )


    return pipeline


# ============================================================
# 11. SUBSET HELPER
#
# Supports pandas Series and numpy arrays safely.
# ============================================================

def subset_vector(
    vector,
    indices,
):

    if hasattr(
        vector,
        "iloc"
    ):

        return vector.iloc[
            indices
        ]

    return np.asarray(
        vector
    )[indices]


# ============================================================
# 12. SINGLE-FOLD EVALUATION
# ============================================================

def evaluate_fold(
    config,
    dataset_id,
    dataset_properties,
    X,
    y,
    train_idx,
    test_idx,
    seed,
):

    # Define these before try so the finally block can safely
    # release them even if pipeline construction or fitting fails.
    X_train = None
    X_test = None
    y_train = None
    y_test = None
    pipeline = None
    probabilities = None
    positive_probability = None

    try:

        # --------------------------------------------------------
        # TRAIN / TEST
        # --------------------------------------------------------

        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]


        y_train = subset_vector(
            y,
            train_idx
        )

        y_test = subset_vector(
            y,
            test_idx
        )


        # --------------------------------------------------------
        # EXPECTED COLUMN POSITIONS
        # --------------------------------------------------------

        expected_columns = set(
            range(
                X.shape[1]
            )
        )


        # --------------------------------------------------------
        # X COLUMN CHECK
        # --------------------------------------------------------

        if (
            set(X_train.columns)
            != expected_columns
        ):

            raise ValueError(
                f"Dataset {dataset_id}: "
                f"X_train columns are "
                f"{set(X_train.columns)}, "
                f"expected "
                f"{expected_columns}."
            )


        if (
            set(X_test.columns)
            != expected_columns
        ):

            raise ValueError(
                f"Dataset {dataset_id}: "
                f"X_test columns are "
                f"{set(X_test.columns)}, "
                f"expected "
                f"{expected_columns}."
            )


        # --------------------------------------------------------
        # FEATURE TYPE CHECK
        # --------------------------------------------------------

        feat_type = (
            dataset_properties[
                "input"
            ]
        )


        if (
            set(feat_type.keys())
            != expected_columns
        ):

            raise ValueError(
                f"Dataset {dataset_id}: "
                f"feat_type keys are "
                f"{set(feat_type.keys())}, "
                f"expected "
                f"{expected_columns}."
            )


        # ========================================================
        # THREAD-LIMITED PIPELINE EXECUTION
        # ========================================================

        with threadpool_limits(
            limits=THREAD_LIMIT
        ):

            # ----------------------------------------------------
            # BUILD
            # ----------------------------------------------------

            pipeline = build_pipeline(
                config=config,

                dataset_id=
                    dataset_id,

                dataset_properties=
                    dataset_properties,

                seed=seed,
            )


            # ----------------------------------------------------
            # FIT
            # ----------------------------------------------------

            pipeline.fit(
                X_train,
                y_train,
            )


            # ----------------------------------------------------
            # PREDICT PROBABILITIES
            # ----------------------------------------------------

            probabilities = (
                pipeline.predict_proba(
                    X_test
                )
            )


            probabilities = np.asarray(
                probabilities
            )


            # ----------------------------------------------------
            # BINARY PROBABILITY CHECK
            # ----------------------------------------------------

            if probabilities.ndim != 2:

                raise ValueError(
                    f"Dataset {dataset_id}: "
                    f"predict_proba returned "
                    f"shape "
                    f"{probabilities.shape}."
                )


            if (
                probabilities.shape[1]
                != 2
            ):

                raise ValueError(
                    f"Dataset {dataset_id}: "
                    f"expected two probability "
                    f"columns, got "
                    f"{probabilities.shape}."
                )


            positive_probability = (
                probabilities[:, 1]
            )


            # ----------------------------------------------------
            # NUMERICAL SAFETY
            # ----------------------------------------------------

            positive_probability = (
                np.asarray(
                    positive_probability,
                    dtype=float,
                )
            )


            positive_probability = (
                np.clip(
                    positive_probability,
                    1e-15,
                    1 - 1e-15,
                )
            )


        # ========================================================
        # LOG LOSS
        # ========================================================

        fold_log_loss = (
            log_loss(
                y_test,
                positive_probability,
                labels=[
                    0,
                    1,
                ],
            )
        )


        return float(
            fold_log_loss
        )

    finally:

        # ========================================================
        # IMPORTANT MEMORY CLEANUP
        #
        # The Auto-sklearn pipeline can hold references to fitted
        # estimators, preprocessors, ConfigSpace objects and
        # intermediate arrays. Explicitly release those references
        # BEFORE the function frame disappears.
        # ========================================================

        pipeline = None
        probabilities = None
        positive_probability = None
        X_train = None
        X_test = None
        y_train = None
        y_test = None

        cleanup_after_execution()


# ============================================================
# 13. CHECKPOINT LOADER
# ============================================================

def load_checkpoint(
    path
):

    if not path.exists():

        return pd.DataFrame()


    try:

        df = pd.read_csv(
            path
        )


        if df.empty:

            return pd.DataFrame()


        return df


    except Exception as e:

        print(
            f"WARNING: Could not read "
            f"checkpoint {path}: "
            f"{type(e).__name__}: {e}"
        )

        return pd.DataFrame()


# ============================================================
# 14. EXECUTION KEY
# ============================================================

def make_execution_key(
    config_id,
    training_seed,
    fold_id,
):

    return (
        str(config_id),
        int(training_seed),
        int(fold_id),
    )


# ============================================================
# 15. REMOVE ONLY THE OLD BOGUS UNIFORMINTEGER ERRORS
# ============================================================
#
# These errors were produced by the PREVIOUS broken Cell 11.
#
# Dataset 37:
#     preserve everything.
#
# Datasets 31,44,4534,1461:
#     remove ONLY errors containing both:
#
#       "Trying to set illegal value"
#       "UniformInteger"
#
# All other errors remain untouched.
# ============================================================

BOGUS_DATASETS = [
    31,
    44,
    4534,
    1461,
]


def remove_bogus_type_errors(
    dataset_id
):

    error_path = (
        CV_ERROR_DIR
        / f"errors_{dataset_id}.csv"
    )


    if not error_path.exists():

        print(
            f"[{dataset_id}] "
            f"No error checkpoint."
        )

        return


    errors = load_checkpoint(
        error_path
    )


    if errors.empty:

        print(
            f"[{dataset_id}] "
            f"Error checkpoint empty."
        )

        return


    if "error" not in errors.columns:

        print(
            f"[{dataset_id}] "
            f"No 'error' column found. "
            f"Checkpoint left untouched."
        )

        return


    error_text = (
        errors[
            "error"
        ]
        .fillna("")
        .astype(str)
    )


    bogus_mask = (
        error_text.str.contains(
            "Trying to set illegal value",
            regex=False,
        )
        &
        error_text.str.contains(
            "UniformInteger",
            regex=False,
        )
    )


    bogus_count = int(
        bogus_mask.sum()
    )


    if bogus_count == 0:

        print(
            f"[{dataset_id}] "
            f"No old bogus UniformInteger "
            f"errors found."
        )

        return


    cleaned_errors = (
        errors.loc[
            ~bogus_mask
        ]
        .copy()
    )


    if cleaned_errors.empty:

        error_path.unlink()

        print(
            f"[{dataset_id}] "
            f"Removed {bogus_count} bogus "
            f"errors. Checkpoint deleted."
        )


    else:

        cleaned_errors.to_csv(
            error_path,
            index=False,
        )

        print(
            f"[{dataset_id}] "
            f"Removed {bogus_count} bogus "
            f"errors. Preserved "
            f"{len(cleaned_errors)} genuine errors."
        )


# ============================================================
# 16. CLEAN OLD CHECKPOINTS
# ============================================================

print()
print("=" * 80)
print("CLEANING OLD INVALID CONFIGSPACE CHECKPOINTS")
print("=" * 80)

for dataset_id in (
    BOGUS_DATASETS
):

    remove_bogus_type_errors(
        dataset_id
    )


# ============================================================
# 17. PRE-CHECK — DATA REPRESENTATION
# ============================================================

print()
print("=" * 80)
print("PRE-CHECK — AUTO-SKLEARN DATA REPRESENTATION")
print("=" * 80)

for dataset_id in DATASET_IDS:

    (
        X_auto,
        y_auto,
        feat_type,
    ) = prepare_autosklearn_data(
        dataset_id
    )


    expected_columns = set(
        range(
            X_auto.shape[1]
        )
    )


    assert (
        set(X_auto.columns)
        == expected_columns
    )


    assert (
        set(feat_type.keys())
        == expected_columns
    )


    n_cat = sum(
        v == "categorical"
        for v in feat_type.values()
    )


    n_num = sum(
        v == "numerical"
        for v in feat_type.values()
    )


    print()
    print(
        f"Dataset {dataset_id}"
    )

    print(
        f"  X shape       : "
        f"{X_auto.shape}"
    )

    print(
        f"  X columns     : "
        f"{list(X_auto.columns)}"
    )

    print(
        f"  Numerical     : "
        f"{n_num}"
    )

    print(
        f"  Categorical   : "
        f"{n_cat}"
    )

    print(
        f"  Feature types : "
        f"{feat_type}"
    )


print()
print(
    "✓ AUTO-SKLEARN DATA REPRESENTATION PASSED."
)


# ============================================================
# 18. PRE-CHECK — CONFIGURATION + PIPELINE
#
# Dataset 31 / config_00000 is checked before any full
# experiment execution.
# ============================================================

print()
print("=" * 80)
print("PRE-CHECK — DATASET 31 / CONFIG_00000")
print("=" * 80)

test_dataset_id = 31


(
    test_X,
    test_y,
    test_feat_type,
) = prepare_autosklearn_data(
    test_dataset_id
)


test_dataset_properties = (
    make_dataset_properties(
        test_dataset_id
    )
)


test_config = (
    CONFIG_OBJECTS[
        "config_00000"
    ]
)


# ------------------------------------------------------------
# Construct target ConfigSpace
# ------------------------------------------------------------

test_target_cs = (
    get_pipeline_config_space(
        dataset_id=
            test_dataset_id,

        dataset_properties=
            test_dataset_properties,
    )
)


print(
    f"Target ConfigSpace hyperparameters: "
    f"{len(test_target_cs.get_hyperparameters())}"
)


# ------------------------------------------------------------
# Normalize config
# ------------------------------------------------------------

(
    test_normalized_config,
    test_dropped,
) = normalize_configuration_types(
    config=
        test_config,

    dataset_id=
        test_dataset_id,

    dataset_properties=
        test_dataset_properties,
)


print(
    f"Original config parameters : "
    f"{len(test_config.get_dictionary())}"
)


print(
    f"Normalized config parameters: "
    f"{len(test_normalized_config.get_dictionary())}"
)


print(
    f"Inactive parameters omitted : "
    f"{len(test_dropped)}"
)


# ------------------------------------------------------------
# Build pipeline
# ------------------------------------------------------------

test_pipeline = build_pipeline(
    config=
        test_config,

    dataset_id=
        test_dataset_id,

    dataset_properties=
        test_dataset_properties,

    seed=
        GLOBAL_SEED,
)


print()
print(
    "✓ Target ConfigSpace constructed."
)

print(
    "✓ Configuration normalized."
)

print(
    "✓ Pipeline constructed."
)

print(
    "✓ Integer-indexed feat_type supplied."
)


# ============================================================
# 19. PRE-CHECK — ACTUAL FIT
#
# This is deliberately one fold only.
#
# If this succeeds, the exact previous error path has been
# eliminated before the 24,750-fold dataset loop starts.
# ============================================================

print()
print("=" * 80)
print("PRE-CHECK — ACTUAL DATASET 31 FIT")
print("=" * 80)


test_fold = (
    CV_SPLITS[
        test_dataset_id
    ][0]
)


test_train_idx = (
    test_fold[
        "train_idx"
    ]
)


test_test_idx = (
    test_fold[
        "test_idx"
    ]
)


print(
    f"Train samples : "
    f"{len(test_train_idx)}"
)


print(
    f"Test samples  : "
    f"{len(test_test_idx)}"
)


print(
    "Running one actual fit/predict..."
)


test_log_loss = (
    evaluate_fold(
        config=
            test_config,

        dataset_id=
            test_dataset_id,

        dataset_properties=
            test_dataset_properties,

        X=
            test_X,

        y=
            test_y,

        train_idx=
            test_train_idx,

        test_idx=
            test_test_idx,

        seed=
            GLOBAL_SEED,
    )
)


print()
print(
    f"✓ ACTUAL PRE-CHECK PASSED."
)

print(
    f"Test fold log-loss: "
    f"{test_log_loss:.8f}"
)


# ============================================================
# 20. FULL EXPERIMENT
# ============================================================

ALL_DATASET_RESULTS = {}


for dataset_id in DATASET_IDS:

    dataset_name = (
        DATASETS[
            dataset_id
        ][
            "name"
        ]
    )


    print("\n\n")
    print("=" * 80)

    print(
        f"DATASET {dataset_id} — "
        f"{dataset_name}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # DATA
    #
    # IMPORTANT:
    # Use PREPARED X, not DATASETS[id]["X"] directly.
    # --------------------------------------------------------

    (
        X,
        y,
        feat_type,
    ) = prepare_autosklearn_data(
        dataset_id
    )


    # --------------------------------------------------------
    # DATASET PROPERTIES
    # --------------------------------------------------------

    dataset_properties = (
        make_dataset_properties(
            dataset_id
        )
    )


    # --------------------------------------------------------
    # CHECKPOINT PATHS
    # --------------------------------------------------------

    result_path = (
        CV_RESULT_DIR
        / f"cv_results_{dataset_id}.csv"
    )


    error_path = (
        CV_ERROR_DIR
        / f"errors_{dataset_id}.csv"
    )


    # --------------------------------------------------------
    # LOAD CHECKPOINTS
    # --------------------------------------------------------

    existing_results = (
        load_checkpoint(
            result_path
        )
    )


    existing_errors = (
        load_checkpoint(
            error_path
        )
    )


    # --------------------------------------------------------
    # DATASET-SPECIFIC CONFIGSPACE
    # --------------------------------------------------------

    target_config_space = (
        get_pipeline_config_space(
            dataset_id=
                dataset_id,

            dataset_properties=
                dataset_properties,
        )
    )


    print(
        f"Target ConfigSpace ready."
    )


    print(
        f"Hyperparameters: "
        f"{len(target_config_space.get_hyperparameters())}"
    )


    # --------------------------------------------------------
    # FIXED CV SPLITS
    # --------------------------------------------------------

    folds = CV_SPLITS[
        dataset_id
    ]


    if len(folds) != N_CV_FOLDS:

        raise ValueError(
            f"Dataset {dataset_id}: "
            f"expected {N_CV_FOLDS} folds, "
            f"got {len(folds)}."
        )


    # --------------------------------------------------------
    # EXPECTED EXECUTIONS
    # --------------------------------------------------------

    total_folds = (
        len(configurations_df)
        *
        len(TRAINING_SEEDS)
        *
        N_CV_FOLDS
    )


    # --------------------------------------------------------
    # COMPLETED EXECUTION KEYS
    # --------------------------------------------------------

    completed_keys = set()


    # --------------------------------------------------------
    # SUCCESSFUL EXECUTIONS
    # --------------------------------------------------------

    if not existing_results.empty:

        required_result_columns = {
            "config_id",
            "training_seed",
            "fold",
        }

        if required_result_columns.issubset(
            existing_results.columns
        ):

            for _, row in (
                existing_results.iterrows()
            ):

                completed_keys.add(
                    make_execution_key(
                        row[
                            "config_id"
                        ],

                        row[
                            "training_seed"
                        ],

                        row[
                            "fold"
                        ],
                    )
                )


    # --------------------------------------------------------
    # GENUINE FAILURES
    # --------------------------------------------------------

    if not existing_errors.empty:

        required_error_columns = {
            "config_id",
            "training_seed",
            "fold",
        }

        if required_error_columns.issubset(
            existing_errors.columns
        ):

            for _, row in (
                existing_errors.iterrows()
            ):

                completed_keys.add(
                    make_execution_key(
                        row[
                            "config_id"
                        ],

                        row[
                            "training_seed"
                        ],

                        row[
                            "fold"
                        ],
                    )
                )


    # --------------------------------------------------------
    # DATASET START
    # --------------------------------------------------------

    dataset_start = time.time()


    print()
    print(
        f"Existing successful folds : "
        f"{len(existing_results)}"
    )

    print(
        f"Existing genuine failures : "
        f"{len(existing_errors)}"
    )

    print(
        f"Unique completed          : "
        f"{len(completed_keys)}"
    )

    print(
        f"Expected executions       : "
        f"{total_folds}"
    )

    print(
        f"Remaining executions      : "
        f"{total_folds - len(completed_keys)}"
    )


    # ========================================================
    # CONFIGURATION LOOP
    # ========================================================

    for config_number, (
        _,
        config_row
    ) in enumerate(
        configurations_df.iterrows(),
        start=1,
    ):

        config_id = (
            config_row[
                "config_id"
            ]
        )


        # ----------------------------------------------------
        # EXISTING NATIVE CONFIGURATION
        # ----------------------------------------------------

        config = (
            CONFIG_OBJECTS[
                config_id
            ]
        )


        # ----------------------------------------------------
        # TRAINING SEEDS
        # ----------------------------------------------------

        for training_seed in (
            TRAINING_SEEDS
        ):


            # ------------------------------------------------
            # FIXED FOLDS
            # ------------------------------------------------

            for fold_info in folds:

                fold_id = (
                    fold_info[
                        "fold"
                    ]
                )


                execution_key = (
                    make_execution_key(
                        config_id=
                            config_id,

                        training_seed=
                            training_seed,

                        fold_id=
                            fold_id,
                    )
                )


                # ============================================
                # RESUME
                # ============================================

                if (
                    execution_key
                    in completed_keys
                ):

                    continue


    


                fold_start = time.time()


                # ============================================
                # EXECUTION
                # ============================================

                try:

                    fold_log_loss = (
                        evaluate_fold(

                            config=
                                config,

                            dataset_id=
                                dataset_id,

                            dataset_properties=
                                dataset_properties,

                            X=
                                X,

                            y=
                                y,

                            train_idx=
                                fold_info[
                                    "train_idx"
                                ],

                            test_idx=
                                fold_info[
                                    "test_idx"
                                ],

                            seed=
                                training_seed,
                        )
                    )


                    # ========================================
                    # SUCCESS ROW
                    # ========================================

                    result_row = {

                        "dataset_id":
                            dataset_id,

                        "dataset":
                            dataset_name,

                        "config_id":
                            config_id,

                        "classifier":
                            config_row[
                                "classifier"
                            ],

                        "feature_preprocessor":
                            config_row[
                                "feature_preprocessor"
                            ],

                        "balancing":
                            config_row[
                                "balancing"
                            ],

                        "config_replicate":
                            config_row[
                                "config_replicate"
                            ],

                        "training_seed":
                            training_seed,

                        "fold":
                            fold_id,

                        "log_loss":
                            fold_log_loss,

                        "status":
                            "success",

                        "runtime_seconds":
                            (
                                time.time()
                                -
                                fold_start
                            ),
                    }


                    # ========================================
                    # IMMEDIATE CHECKPOINT
                    # ========================================

                    pd.DataFrame(
                        [result_row]
                    ).to_csv(

                        result_path,

                        mode="a",

                        header=(
                            not result_path.exists()
                        ),

                        index=False,
                    )


                    completed_keys.add(
                        execution_key
                    )


                # ============================================
                # FAILURE
                # ============================================

                except Exception as e:

                    error_row = {

                        "dataset_id":
                            dataset_id,

                        "dataset":
                            dataset_name,

                        "config_id":
                            config_id,

                        "classifier":
                            config_row[
                                "classifier"
                            ],

                        "feature_preprocessor":
                            config_row[
                                "feature_preprocessor"
                            ],

                        "balancing":
                            config_row[
                                "balancing"
                            ],

                        "config_replicate":
                            config_row[
                                "config_replicate"
                            ],

                        "training_seed":
                            training_seed,

                        "fold":
                            fold_id,

                        "error_type":
                            type(e).__name__,

                        "error":
                            repr(e),

                        "runtime_seconds":
                            (
                                time.time()
                                -
                                fold_start
                            ),

                        "status":
                            "failed",
                    }


                    # ========================================
                    # IMMEDIATE ERROR CHECKPOINT
                    # ========================================

                    pd.DataFrame(
                        [error_row]
                    ).to_csv(

                        error_path,

                        mode="a",

                        header=(
                            not error_path.exists()
                        ),

                        index=False,
                    )


                    completed_keys.add(
                        execution_key
                    )


                    # ========================================
                    # PRINT FAILURE
                    # ========================================

                    print(
                        f"[{dataset_id}] "
                        f"FAIL | "
                        f"{config_id} | "
                        f"{config_row['classifier']} | "
                        f"{config_row['feature_preprocessor']} | "
                        f"{config_row['balancing']} | "
                        f"seed={training_seed} | "
                        f"fold={fold_id} | "
                        f"{type(e).__name__}: "
                        f"{e}"
                    )


                # ============================================
                # PROGRESS
                # ============================================

                completed_count = len(
                    completed_keys
                )


                if (
                    completed_count % 25 == 0
                    or
                    completed_count
                    == total_folds
                ):

                    elapsed = (
                        time.time()
                        -
                        dataset_start
                    )


                    remaining = (
                        total_folds
                        -
                        completed_count
                    )


                    rate = (
                        completed_count
                        /
                        max(
                            elapsed / 60,
                            1e-9,
                        )
                    )


                    eta_minutes = (
                        remaining / rate
                        if rate > 0
                        else np.nan
                    )


                    print(
                        f"[{dataset_id}] "
                        f"{completed_count}/"
                        f"{total_folds} "
                        f"("
                        f"{100 * completed_count / total_folds:.2f}%"
                        f") | "
                        f"config "
                        f"{config_number}/"
                        f"{len(configurations_df)} | "
                        f"elapsed "
                        f"{elapsed / 60:.1f} min | "
                        f"ETA "
                        f"{eta_minutes / 60:.1f} h"
                    )


    # ========================================================
    # DATASET COMPLETE
    # ========================================================

    final_results = (
        load_checkpoint(
            result_path
        )
    )


    final_errors = (
        load_checkpoint(
            error_path
        )
    )


    ALL_DATASET_RESULTS[
        dataset_id
    ] = final_results


    # ========================================================
    # DATASET SUMMARY
    # ========================================================

    successful_count = (
        len(final_results)
    )


    failed_count = (
        len(final_errors)
    )


    attempted_count = (
        successful_count
        +
        failed_count
    )


    print()
    print("-" * 80)

    print(
        f"DATASET {dataset_id} COMPLETE"
    )

    print("-" * 80)

    print(
        f"Successful folds : "
        f"{successful_count}"
    )

    print(
        f"Failed folds     : "
        f"{failed_count}"
    )

    print(
        f"Total attempted  : "
        f"{attempted_count}"
    )

    print(
        f"Expected folds   : "
        f"{total_folds}"
    )


    # ========================================================
    # STRICT COMPLETENESS
    # ========================================================

    if (
        attempted_count
        != total_folds
    ):

        missing = (
            total_folds
            -
            attempted_count
        )

        raise RuntimeError(
            f"DATASET {dataset_id} "
            f"IS NOT COMPLETE. "
            f"Missing {missing} executions."
        )


    print(
        f"✓ Dataset {dataset_id} "
        f"has exactly "
        f"{total_folds} "
        f"attempted executions."
    )


# ============================================================
# 21. GLOBAL SUMMARY
# ============================================================

print("\n\n")
print("=" * 80)
print("FULL CV EXPERIMENT COMPLETE")
print("=" * 80)


grand_success = 0
grand_failure = 0


for dataset_id in DATASET_IDS:

    dataset_name = (
        DATASETS[
            dataset_id
        ][
            "name"
        ]
    )


    result_path = (
        CV_RESULT_DIR
        /
        f"cv_results_{dataset_id}.csv"
    )


    error_path = (
        CV_ERROR_DIR
        /
        f"errors_{dataset_id}.csv"
    )


    results = (
        load_checkpoint(
            result_path
        )
    )


    errors = (
        load_checkpoint(
            error_path
        )
    )


    success_count = len(
        results
    )


    failure_count = len(
        errors
    )


    attempted = (
        success_count
        +
        failure_count
    )


    grand_success += (
        success_count
    )


    grand_failure += (
        failure_count
    )


    print(
        f"{dataset_id} | "
        f"{dataset_name:20s} | "
        f"{success_count:6d} successful | "
        f"{failure_count:6d} failed | "
        f"{attempted:6d} attempted"
    )


# ============================================================
# GLOBAL TOTALS
# ============================================================

grand_total = (
    grand_success
    +
    grand_failure
)


grand_expected = (
    len(DATASET_IDS)
    *
    len(configurations_df)
    *
    len(TRAINING_SEEDS)
    *
    N_CV_FOLDS
)


print("-" * 80)


print(
    f"TOTAL SUCCESSFUL : "
    f"{grand_success}"
)


print(
    f"TOTAL FAILED     : "
    f"{grand_failure}"
)


print(
    f"TOTAL ATTEMPTED  : "
    f"{grand_total}"
)


print(
    f"TOTAL EXPECTED   : "
    f"{grand_expected}"
)


print("-" * 80)


assert (
    grand_total
    ==
    grand_expected
), (
    f"Global completeness failure: "
    f"{grand_total} attempted vs "
    f"{grand_expected} expected."
)


print(
    "✓ ALL DATASETS HAVE THE EXPECTED "
    "NUMBER OF ATTEMPTED EXECUTIONS."
)


print(
    "✓ CV EXPERIMENT FINISHED."
)

CONFIGURATION OBJECT VERIFICATION
Configuration objects : 1650
Configuration dataframe: 1650
✓ Existing native ConfigSpace configurations recovered.

DATASET / FEATURE-TYPE VERIFICATION

Dataset 37
  Shape        : (768, 8)
  Columns      : [0, 1, 2, 3, 4, 5, 6, 7]
  Numerical    : 8
  Categorical  : 0
  feat_type    : {0: 'numerical', 1: 'numerical', 2: 'numerical', 3: 'numerical', 4: 'numerical', 5: 'numerical', 6: 'numerical', 7: 'numerical'}

Dataset 31
  Shape        : (1000, 20)
  Columns      : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  Numerical    : 7
  Categorical  : 13
  feat_type    : {0: 'categorical', 1: 'numerical', 2: 'categorical', 3: 'categorical', 4: 'numerical', 5: 'categorical', 6: 'categorical', 7: 'numerical', 8: 'categorical', 9: 'categorical', 10: 'numerical', 11: 'categorical', 12: 'numerical', 13: 'categorical', 14: 'categorical', 15: 'numerical', 16: 'categorical', 17: 'numerical', 18: 'categorical', 19: 'categorical'}

Dataset 


✓ Target ConfigSpace constructed.
✓ Configuration normalized.
✓ Pipeline constructed.
✓ Integer-indexed feat_type supplied.

PRE-CHECK — ACTUAL DATASET 31 FIT
Train samples : 800
Test samples  : 200
Running one actual fit/predict...

✓ ACTUAL PRE-CHECK PASSED.
Test fold log-loss: 3.64533342



DATASET 37 — diabetes
Target ConfigSpace ready.
Hyperparameters: 165

Existing successful folds : 24570
Existing genuine failures : 180
Unique completed          : 24750
Expected executions       : 24750
Remaining executions      : 0

--------------------------------------------------------------------------------
DATASET 37 COMPLETE
--------------------------------------------------------------------------------
Successful folds : 24570
Failed folds     : 180
Total attempted  : 24750
Expected folds   : 24750
✓ Dataset 37 has exactly 24750 attempted executions.



DATASET 31 — credit-g

Target ConfigSpace ready.
Hyperparameters: 168

Existing successful folds : 10611
Existing genuine failures : 

KeyboardInterrupt: 